In [1]:
from pathlib import Path
import sys

# ---------------------------------------------------------------------
# Locate project root
# ---------------------------------------------------------------------

project_root = Path.cwd()

while project_root.name != "EventCameraProject":
    if project_root.parent == project_root:
        raise RuntimeError("Could not locate EventCameraProject root.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project Root:", project_root)

Project Root: /home/ayon/git/EventCameraProject


In [5]:
# ============================================================
# FULL TRAINING SMOKE TEST — 2 EPOCHS
# EventCameraProject / EVIMO2
# ============================================================

from pathlib import Path
import time

import torch
from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
    VoxelizeEvents,
)

from src.models.world_model.model import WorldModel
from src.losses.total_loss import TotalLoss


# ============================================================
# Configuration
# ============================================================

NUM_EPOCHS = 2

BATCH_SIZE = 2

NUM_WORKERS = 0

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-5

NUM_BINS = 5

HISTORY_OFFSETS = (-3, -2, -1, 0)

DATASET_ROOT = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 90)
print("TRAINING CONFIGURATION")
print("=" * 90)

print("Device          :", device)

if device.type == "cuda":
    print("GPU             :", torch.cuda.get_device_name(0))
    print(
        "GPU Memory      :",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB",
    )

print("Epochs          :", NUM_EPOCHS)
print("Batch size      :", BATCH_SIZE)
print("Learning rate   :", LEARNING_RATE)
print("Weight decay    :", WEIGHT_DECAY)
print("Voxel bins      :", NUM_BINS)
print("History offsets :", HISTORY_OFFSETS)

print()


# ============================================================
# Dataset
# ============================================================

print("=" * 90)
print("CREATING DATASET")
print("=" * 90)

frame_dataset = EVIMO2Dataset(
    dataset_root=DATASET_ROOT,
    sensors=("left_camera", "right_camera"),
    split="train",

    # These are kept because the model/loss pipeline may
    # still use the corresponding metadata.
    #
    # IMPORTANT:
    # We are NOT using temporal mask supervision.
    load_depth=True,
    load_mask=True,
)

temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=HISTORY_OFFSETS,
)

print("Frame samples   :", len(frame_dataset))
print("Temporal samples:", len(temporal_dataset))

print()


# ============================================================
# DataLoader
# ============================================================

loader = DataLoader(
    temporal_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    collate_fn=temporal_collate_fn,

    pin_memory=(device.type == "cuda"),
)

print("DataLoader ready.")
print()


# ============================================================
# Transform pipeline
# ============================================================

transform = Compose(
    [
        ToTensor(),

        NormalizeEventTime(),

        NormalizeIMU(),

        VoxelizeEvents(
            num_bins=NUM_BINS,
        ),
    ]
)

print("Transform pipeline ready.")
print()


# ============================================================
# Model
# ============================================================

print("=" * 90)
print("CREATING MODEL")
print("=" * 90)

model = WorldModel().to(device)

model.train()

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    ),
)

print()


# ============================================================
# Total Loss
# ============================================================

print("=" * 90)
print("CREATING LOSS")
print("=" * 90)

loss_fn = TotalLoss(

    latent_weight=1.0,

    depth_smoothness_weight=1.0,

    pose_temporal_weight=1.0,

    depth_temporal_weight=1.0,

    dynamic_mask_weight=1.0,
)

loss_fn = loss_fn.to(device)

print("TotalLoss ready.")
print()


# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,
)


# ============================================================
# AMP
# ============================================================

# AMP is useful for the RTX 4070 SUPER and can substantially
# reduce memory consumption for the voxel/depth network.

use_amp = device.type == "cuda"

if use_amp:

    scaler = torch.amp.GradScaler(
        "cuda"
    )

else:

    scaler = None


print("=" * 90)
print("TRAINING START")
print("=" * 90)

print()


# ============================================================
# Training
# ============================================================

for epoch in range(NUM_EPOCHS):

    epoch_start = time.time()

    model.train()

    # --------------------------------------------------------
    # Running statistics
    # --------------------------------------------------------

    running_total = 0.0

    running_latent = 0.0

    running_depth_smoothness = 0.0

    running_pose_temporal = 0.0

    running_depth_temporal = 0.0

    running_dynamic_mask = 0.0

    running_prediction = 0.0

    running_rendering = 0.0

    running_agreement = 0.0

    running_mask_sparsity = 0.0

    running_mask_confidence = 0.0

    running_dynamic_ratio = 0.0

    num_batches = 0

    print("=" * 90)
    print(f"EPOCH {epoch + 1}/{NUM_EPOCHS}")
    print("=" * 90)

    # --------------------------------------------------------
    # Iterate over COMPLETE dataset
    # --------------------------------------------------------

    for batch_idx, raw_batch in enumerate(loader):

        batch_start = time.time()

        # ====================================================
        # Transform
        # ====================================================

        voxel_batch = transform(raw_batch)

        # ====================================================
        # Move COMPLETE temporal batch to GPU
        # ====================================================

        voxel_batch = voxel_batch.to(device)

        # ====================================================
        # Build voxel tensor
        #
        # Shape:
        #
        # (B, T, bins, H, W)
        # ====================================================

        voxels = torch.stack(
            [
                frame.voxel_grid
                for frame in voxel_batch.frames
            ],
            dim=1,
        )

        # ----------------------------------------------------
        # Safety checks
        # ----------------------------------------------------

        assert voxels.ndim == 5

        assert voxels.device.type == device.type, (
            f"Voxels device mismatch: "
            f"got {voxels.device}, expected {device}"
        )

        # ====================================================
        # Zero gradients
        # ====================================================

        optimizer.zero_grad(
            set_to_none=True
        )

        # ====================================================
        # Forward + Loss
        # ====================================================

        if use_amp:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):

                outputs = model(
                    voxels,
                    voxel_batch,
                )

                loss_output = loss_fn(
                    outputs=outputs
                )

                total_loss = loss_output["loss"]

        else:

            outputs = model(
                voxels,
                voxel_batch,
            )

            loss_output = loss_fn(
                outputs=outputs
            )

            total_loss = loss_output["loss"]

        # ====================================================
        # Loss validity
        # ====================================================

        if not torch.isfinite(total_loss):

            print()
            print("!" * 90)
            print("NON-FINITE LOSS DETECTED")
            print("Epoch :", epoch + 1)
            print("Batch :", batch_idx)
            print("Loss  :", total_loss)
            print("!" * 90)

            raise RuntimeError(
                "Training stopped because total loss is not finite."
            )

        # ====================================================
        # Backward
        # ====================================================

        if use_amp:

            scaler.scale(
                total_loss
            ).backward()

            # ------------------------------------------------
            # Unscale before gradient clipping/checking
            # ------------------------------------------------

            scaler.unscale_(
                optimizer
            )

        else:

            total_loss.backward()

        # ====================================================
        # Gradient validity
        # ====================================================

        gradient_parameters = 0

        for name, parameter in model.named_parameters():

            if parameter.grad is not None:

                if not torch.isfinite(
                    parameter.grad
                ).all():

                    print(
                        "Invalid gradient in:",
                        name,
                    )

                    raise RuntimeError(
                        "Non-finite gradient detected."
                    )

                gradient_parameters += 1

        # ====================================================
        # Gradient clipping
        # ====================================================

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        # ====================================================
        # Optimizer step
        # ====================================================

        if use_amp:

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            optimizer.step()

        # ====================================================
        # Accumulate statistics
        # ====================================================

        running_total += (
            total_loss.detach().item()
        )

        running_latent += (
            loss_output["latent_loss"]
            .detach()
            .item()
        )

        running_depth_smoothness += (
            loss_output["depth_smoothness_loss"]
            .detach()
            .item()
        )

        running_pose_temporal += (
            loss_output["pose_temporal_loss"]
            .detach()
            .item()
        )

        running_depth_temporal += (
            loss_output["depth_temporal_loss"]
            .detach()
            .item()
        )

        running_dynamic_mask += (
            loss_output["dynamic_mask_loss"]
            .detach()
            .item()
        )

        running_prediction += (
            loss_output["prediction_loss"]
            .detach()
            .item()
        )

        running_rendering += (
            loss_output["rendering_loss"]
            .detach()
            .item()
        )

        running_agreement += (
            loss_output["agreement_loss"]
            .detach()
            .item()
        )

        running_mask_sparsity += (
            loss_output["mask_sparsity_loss"]
            .detach()
            .item()
        )

        running_mask_confidence += (
            loss_output["mask_confidence_loss"]
            .detach()
            .item()
        )

        running_dynamic_ratio += (
            loss_output["dynamic_ratio"]
            .detach()
            .item()
        )

        num_batches += 1

        # ====================================================
        # Progress
        # ====================================================

        batch_time = time.time() - batch_start

        if (
            batch_idx % 10 == 0
            or batch_idx == len(loader) - 1
        ):

            print(
                f"[Epoch {epoch + 1}/{NUM_EPOCHS}] "
                f"[Batch {batch_idx + 1}/{len(loader)}] "
                f"Loss: {total_loss.item():.6f} "
                f"Time: {batch_time:.2f}s "
                f"GradParams: {gradient_parameters}"
            )

    # ========================================================
    # Epoch averages
    # ========================================================

    avg_total = (
        running_total / num_batches
    )

    avg_latent = (
        running_latent / num_batches
    )

    avg_depth_smoothness = (
        running_depth_smoothness / num_batches
    )

    avg_pose_temporal = (
        running_pose_temporal / num_batches
    )

    avg_depth_temporal = (
        running_depth_temporal / num_batches
    )

    avg_dynamic_mask = (
        running_dynamic_mask / num_batches
    )

    avg_prediction = (
        running_prediction / num_batches
    )

    avg_rendering = (
        running_rendering / num_batches
    )

    avg_agreement = (
        running_agreement / num_batches
    )

    avg_mask_sparsity = (
        running_mask_sparsity / num_batches
    )

    avg_mask_confidence = (
        running_mask_confidence / num_batches
    )

    avg_dynamic_ratio = (
        running_dynamic_ratio / num_batches
    )

    epoch_time = (
        time.time() - epoch_start
    )

    # ========================================================
    # Epoch summary
    # ========================================================

    print()
    print("-" * 90)
    print(
        f"EPOCH {epoch + 1} SUMMARY"
    )
    print("-" * 90)

    print(
        f"Total Loss              : "
        f"{avg_total:.6f}"
    )

    print()

    print(
        f"Latent Loss             : "
        f"{avg_latent:.6f}"
    )

    print(
        f"  Prediction Loss       : "
        f"{avg_prediction:.6f}"
    )

    print(
        f"  Rendering Loss        : "
        f"{avg_rendering:.6f}"
    )

    print(
        f"  Agreement Loss        : "
        f"{avg_agreement:.6f}"
    )

    print()

    print(
        f"Depth Smoothness        : "
        f"{avg_depth_smoothness:.6f}"
    )

    print(
        f"Depth Temporal          : "
        f"{avg_depth_temporal:.6f}"
    )

    print(
        f"Pose Temporal           : "
        f"{avg_pose_temporal:.6f}"
    )

    print()

    print(
        f"Dynamic Mask Loss       : "
        f"{avg_dynamic_mask:.6f}"
    )

    print(
        f"  Mask Sparsity         : "
        f"{avg_mask_sparsity:.6f}"
    )

    print(
        f"  Mask Confidence       : "
        f"{avg_mask_confidence:.6f}"
    )

    print(
        f"  Dynamic Ratio         : "
        f"{avg_dynamic_ratio:.4f}"
    )

    print()

    print(
        f"Batches                 : "
        f"{num_batches}"
    )

    print(
        f"Epoch Time              : "
        f"{epoch_time / 60:.2f} min"
    )

    if device.type == "cuda":

        print(
            f"Peak GPU Memory        : "
            f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
        )

        torch.cuda.reset_peak_memory_stats()

    print("-" * 90)

    print()


# ============================================================
# Final state
# ============================================================

print("=" * 90)
print("✓ 2-EPOCH TRAINING COMPLETED")
print("=" * 90)

print(
    "Final learning rate:",
    optimizer.param_groups[0]["lr"],
)

if device.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

print("=" * 90)

TRAINING CONFIGURATION
Device          : cuda
GPU             : NVIDIA GeForce RTX 4070 SUPER
GPU Memory      : 11.59 GB
Epochs          : 2
Batch size      : 2
Learning rate   : 0.0001
Weight decay    : 1e-05
Voxel bins      : 5
History offsets : (-3, -2, -1, 0)

CREATING DATASET
EVIMO2 Sequence Index
Sequences : 22
Frames    : 9352
Sensors   : left_camera, right_camera
Split     : train
Frame samples   : 9352
Temporal samples: 9286

DataLoader ready.

Transform pipeline ready.

CREATING MODEL
Trainable parameters: 15073368

CREATING LOSS
TotalLoss ready.

TRAINING START

EPOCH 1/2
jij
Invalid gradient in: event_encoder.stem.0.block.0.weight


RuntimeError: Non-finite gradient detected.

In [7]:
# ============================================================
# FULL NON-FINITE GRADIENT DIAGNOSTIC
# ============================================================
#
# Purpose:
#   Diagnose NaN/Inf gradients during the first training step.
#
# This cell:
#   1. Creates the EVIMO2 datasets
#   2. Creates the temporal DataLoader
#   3. Creates the transform pipeline
#   4. Moves the complete temporal batch to GPU
#   5. Creates the WorldModel
#   6. Creates TotalLoss
#   7. Runs one forward pass
#   8. Checks every model output for NaN/Inf
#   9. Checks every loss component
#  10. Runs backward with anomaly detection
#  11. Finds the first parameter with invalid gradients
#  12. Prints gradient statistics
#
# IMPORTANT:
#   This does NOT perform an optimizer step.
# ============================================================

from pathlib import Path
import torch
from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
    VoxelizeEvents,
)

from src.models.world_model.model import WorldModel
from src.losses.total_loss import TotalLoss


# ============================================================
# CONFIGURATION
# ============================================================

DATASET_ROOT = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

BATCH_SIZE = 2

VOXEL_BINS = 5

HISTORY_OFFSETS = (-3, -2, -1, 0)

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-5


# ============================================================
# DEVICE CHECK
# ============================================================

print("=" * 90)
print("DEVICE")
print("=" * 90)

print("Device:", DEVICE)

if DEVICE.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    props = torch.cuda.get_device_properties(0)

    print(
        "GPU memory:",
        round(props.total_memory / 1024**3, 2),
        "GB",
    )

else:

    print("WARNING: CUDA is not available.")


# ============================================================
# DATASET
# ============================================================

print()
print("=" * 90)
print("CREATING DATASET")
print("=" * 90)


frame_dataset = EVIMO2Dataset(
    dataset_root=DATASET_ROOT,
    sensors=(
        "left_camera",
        "right_camera",
    ),
    split="train",
    load_depth=True,
    load_mask=True,
)


temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=HISTORY_OFFSETS,
)


loader = DataLoader(
    temporal_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=temporal_collate_fn,
)


print(
    "Frame samples   :",
    len(frame_dataset),
)

print(
    "Temporal samples:",
    len(temporal_dataset),
)

print(
    "Batch size      :",
    BATCH_SIZE,
)


# ============================================================
# TRANSFORMS
# ============================================================

print()
print("=" * 90)
print("TRANSFORM PIPELINE")
print("=" * 90)


transform = Compose(
    [
        ToTensor(),

        NormalizeEventTime(),

        NormalizeIMU(),

        VoxelizeEvents(
            num_bins=VOXEL_BINS,
        ),
    ]
)


print("Voxel bins:", VOXEL_BINS)


# ============================================================
# GET ONE BATCH
# ============================================================

print()
print("=" * 90)
print("LOADING ONE BATCH")
print("=" * 90)


raw_batch = next(iter(loader))

voxel_batch = transform(
    raw_batch
)


print(
    "Temporal frames:",
    len(voxel_batch.frames),
)


# ============================================================
# MOVE COMPLETE TEMPORAL BATCH TO GPU
# ============================================================

print()
print("=" * 90)
print("MOVING TEMPORAL BATCH TO DEVICE")
print("=" * 90)


voxel_batch = voxel_batch.to(
    DEVICE
)


for i, frame in enumerate(
    voxel_batch.frames
):

    print(
        f"Frame {i}:",
        "voxel=",
        frame.voxel_grid.device,
        "imu=",
        frame.imu_angular_velocity.device,
    )


# ============================================================
# BUILD VOXEL TENSOR
# ============================================================

voxels = torch.stack(
    [
        frame.voxel_grid
        for frame in voxel_batch.frames
    ],
    dim=1,
)


print()
print("=" * 90)
print("VOXELS")
print("=" * 90)

print(
    "Shape :",
    tuple(voxels.shape),
)

print(
    "dtype :",
    voxels.dtype,
)

print(
    "device:",
    voxels.device,
)


assert voxels.ndim == 5

assert voxels.device.type == DEVICE.type


# ============================================================
# CHECK INPUT FOR NaN / INF
# ============================================================

print()
print("=" * 90)
print("INPUT FINITE CHECK")
print("=" * 90)


def check_tensor(
    name,
    tensor,
):

    if not torch.is_tensor(tensor):

        print(
            f"{name:<45} NOT A TENSOR"
        )

        return True


    finite = torch.isfinite(
        tensor
    ).all().item()


    if not finite:

        nan_count = torch.isnan(
            tensor
        ).sum().item()

        inf_count = torch.isinf(
            tensor
        ).sum().item()

        print(
            f"INVALID: {name}"
        )

        print(
            "  shape:",
            tuple(tensor.shape)
        )

        print(
            "  nan:",
            nan_count
        )

        print(
            "  inf:",
            inf_count
        )

        return False


    print(
        f"OK: {name:<45}",
        f"min={tensor.min().item():.6e}",
        f"max={tensor.max().item():.6e}",
        f"mean={tensor.mean().item():.6e}",
    )

    return True


check_tensor(
    "voxels",
    voxels,
)


for i, frame in enumerate(
    voxel_batch.frames
):

    check_tensor(
        f"frame[{i}].voxel_grid",
        frame.voxel_grid,
    )

    check_tensor(
        f"frame[{i}].imu_angular_velocity",
        frame.imu_angular_velocity,
    )

    check_tensor(
        f"frame[{i}].imu_linear_acceleration",
        frame.imu_linear_acceleration,
    )

    check_tensor(
        f"frame[{i}].imu_timestamps",
        frame.imu_timestamps,
    )

    check_tensor(
        f"frame[{i}].camera_intrinsics",
        frame.camera_intrinsics,
    )

    check_tensor(
        f"frame[{i}].camera_distortion",
        frame.camera_distortion,
    )


# ============================================================
# MODEL
# ============================================================

print()
print("=" * 90)
print("CREATING MODEL")
print("=" * 90)


model = WorldModel().to(
    DEVICE
)

model.train()


num_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    "Trainable parameters:",
    num_parameters,
)


# ============================================================
# CHECK MODEL PARAMETERS
# ============================================================

print()
print("=" * 90)
print("MODEL PARAMETER CHECK")
print("=" * 90)


invalid_parameters = []


for name, parameter in model.named_parameters():

    if not torch.isfinite(
        parameter
    ).all():

        invalid_parameters.append(
            name
        )


if invalid_parameters:

    print(
        "INVALID MODEL PARAMETERS:"
    )

    for name in invalid_parameters:

        print(
            "  ",
            name
        )

    raise RuntimeError(
        "Model already contains NaN/Inf parameters."
    )


print(
    "All model parameters are finite."
)


# ============================================================
# TOTAL LOSS
# ============================================================

print()
print("=" * 90)
print("CREATING TOTAL LOSS")
print("=" * 90)


loss_fn = TotalLoss(
    latent_weight=1.0,
    depth_smoothness_weight=1.0,
    pose_temporal_weight=1.0,
    depth_temporal_weight=1.0,
    dynamic_mask_weight=1.0,
)


loss_fn = loss_fn.to(
    DEVICE
)


print(
    "TotalLoss ready."
)


# ============================================================
# FORWARD PASS
# ============================================================

print()
print("=" * 90)
print("FORWARD PASS")
print("=" * 90)


print("Running model...")


with torch.autograd.detect_anomaly():

    outputs = model(
        voxels,
        voxel_batch,
    )


print(
    "Forward pass completed."
)


# ============================================================
# CHECK ALL MODEL OUTPUTS
# ============================================================

print()
print("=" * 90)
print("MODEL OUTPUT FINITE CHECK")
print("=" * 90)


for name, value in outputs.items():

    if torch.is_tensor(value):

        check_tensor(
            f"outputs['{name}']",
            value,
        )

    elif isinstance(value, dict):

        print(
            f"\noutputs['{name}'] -> dict"
        )

        for subname, subvalue in value.items():

            if torch.is_tensor(subvalue):

                check_tensor(
                    f"outputs['{name}']['{subname}']",
                    subvalue,
                )

    else:

        print(
            f"{name:<45}",
            type(value),
        )


# ============================================================
# LOSS FORWARD
# ============================================================

print()
print("=" * 90)
print("LOSS FORWARD")
print("=" * 90)


with torch.autograd.detect_anomaly():

    loss_output = loss_fn(
        outputs=outputs
    )


print(
    "Loss computation completed."
)


# ============================================================
# PRINT LOSS COMPONENTS
# ============================================================

print()
print("=" * 90)
print("LOSS BREAKDOWN")
print("=" * 90)


for name, value in loss_output.items():

    if torch.is_tensor(value):

        finite = torch.isfinite(
            value
        ).item()

        if value.ndim == 0:

            print(
                f"{name:<30}",
                f"{value.item():.8f}",
                "FINITE" if finite else "INVALID",
            )

        else:

            print(
                f"{name:<30}",
                "shape=",
                tuple(value.shape),
                "FINITE" if finite else "INVALID",
            )

    else:

        print(
            f"{name:<30}",
            value,
        )


total_loss = loss_output["loss"]


# ============================================================
# TOTAL LOSS CHECK
# ============================================================

print()
print("=" * 90)
print("TOTAL LOSS CHECK")
print("=" * 90)


print(
    "Total loss:",
    total_loss.item(),
)


if not torch.isfinite(
    total_loss
):

    raise RuntimeError(
        "Total loss is already NaN/Inf before backward."
    )


# ============================================================
# REGISTER GRADIENT HOOKS
# ============================================================
#
# These hooks tell us which parameter receives the first
# invalid gradient.
# ============================================================

print()
print("=" * 90)
print("REGISTERING GRADIENT HOOKS")
print("=" * 90)


gradient_status = {}


def make_gradient_hook(
    parameter_name
):

    def hook(
        grad
    ):

        if grad is None:

            gradient_status[
                parameter_name
            ] = {
                "valid": False,
                "reason": "None",
            }

            return grad


        finite = torch.isfinite(
            grad
        ).all().item()


        if not finite:

            nan_count = torch.isnan(
                grad
            ).sum().item()

            inf_count = torch.isinf(
                grad
            ).sum().item()


            gradient_status[
                parameter_name
            ] = {

                "valid": False,

                "reason": "NaN/Inf",

                "nan": nan_count,

                "inf": inf_count,

                "shape": tuple(
                    grad.shape
                ),

            }


        else:

            gradient_status[
                parameter_name
            ] = {

                "valid": True,

                "min": grad.min().item(),

                "max": grad.max().item(),

                "mean": grad.mean().item(),

                "norm": grad.norm().item(),

            }


        return grad


    return hook


for name, parameter in model.named_parameters():

    if parameter.requires_grad:

        parameter.register_hook(
            make_gradient_hook(name)
        )


# ============================================================
# BACKWARD
# ============================================================

print()
print("=" * 90)
print("BACKWARD PASS")
print("=" * 90)


model.zero_grad(
    set_to_none=True
)


try:

    with torch.autograd.detect_anomaly():

        total_loss.backward()

    backward_failed = False

    print(
        "Backward completed."
    )


except RuntimeError as error:

    backward_failed = True

    print()
    print(
        "!!! BACKWARD FAILED !!!"
    )

    print(
        error
    )


# ============================================================
# GRADIENT ANALYSIS
# ============================================================

print()
print("=" * 90)
print("GRADIENT ANALYSIS")
print("=" * 90)


invalid_gradients = []


for name, parameter in model.named_parameters():

    if not parameter.requires_grad:

        continue


    grad = parameter.grad


    if grad is None:

        print(
            f"{name:<60}",
            "NO GRAD",
        )

        continue


    finite = torch.isfinite(
        grad
    ).all().item()


    if not finite:

        nan_count = torch.isnan(
            grad
        ).sum().item()

        inf_count = torch.isinf(
            grad
        ).sum().item()


        invalid_gradients.append(
            name
        )


        print(
            f"INVALID: {name}"
        )

        print(
            "   shape:",
            tuple(grad.shape)
        )

        print(
            "   NaN:",
            nan_count
        )

        print(
            "   Inf:",
            inf_count
        )


    else:

        print(
            f"OK: {name:<60}",
            f"norm={grad.norm().item():.6e}"
        )


# ============================================================
# FIRST INVALID GRADIENT
# ============================================================

print()
print("=" * 90)
print("DIAGNOSTIC RESULT")
print("=" * 90)


if invalid_gradients:

    print(
        "NON-FINITE GRADIENTS FOUND."
    )

    print()

    print(
        "Number of invalid parameters:",
        len(invalid_gradients),
    )

    print()

    print(
        "First invalid parameter:"
    )

    print(
        "   ",
        invalid_gradients[0]
    )

    print()

    print(
        "Gradient-hook information:"
    )

    for name, info in gradient_status.items():

        if not info.get(
            "valid",
            True,
        ):

            print(
                name,
                "->",
                info,
            )

            break


else:

    print(
        "✓ ALL PARAMETER GRADIENTS ARE FINITE"
    )


# ============================================================
# CUDA MEMORY
# ============================================================

if DEVICE.type == "cuda":

    print()
    print("=" * 90)
    print("CUDA MEMORY")
    print("=" * 90)

    print(
        "Allocated:",
        round(
            torch.cuda.memory_allocated()
            / 1024**3,
            3,
        ),
        "GB",
    )

    print(
        "Reserved :",
        round(
            torch.cuda.memory_reserved()
            / 1024**3,
            3,
        ),
        "GB",
    )


# ============================================================
# FINAL INTERPRETATION
# ============================================================

print()
print("=" * 90)
print("FINAL DIAGNOSTIC")
print("=" * 90)


if not torch.isfinite(
    total_loss
):

    print(
        "CAUSE CATEGORY:"
    )

    print(
        "Loss/output became NaN/Inf before backward."
    )


elif invalid_gradients:

    print(
        "CAUSE CATEGORY:"
    )

    print(
        "Forward loss is finite, but backward produces"
    )

    print(
        "NaN/Inf gradients."
    )

    print()

    print(
        "Most likely investigation targets:"
    )

    print(
        "1. Event encoder numerical instability"
    )

    print(
        "2. Extremely large voxel/event values"
    )

    print(
        "3. Exploding activations in the encoder"
    )

    print(
        "4. An unstable operation inside one loss"
    )

    print(
        "5. Division/log/sqrt operation without numerical protection"
    )

    print(
        "6. Excessive gradient magnitude"
    )


else:

    print(
        "✓ No NaN/Inf gradients detected."
    )

    print(
        "The previous optimizer-step failure could not be reproduced"
    )

    print(
        "with this batch."
    )

print()
print("=" * 90)
print("END DIAGNOSTIC")
print("=" * 90)

DEVICE
Device: cuda
GPU: NVIDIA GeForce RTX 4070 SUPER
GPU memory: 11.59 GB

CREATING DATASET
EVIMO2 Sequence Index
Sequences : 22
Frames    : 9352
Sensors   : left_camera, right_camera
Split     : train
Frame samples   : 9352
Temporal samples: 9286
Batch size      : 2

TRANSFORM PIPELINE
Voxel bins: 5

LOADING ONE BATCH
Temporal frames: 4

MOVING TEMPORAL BATCH TO DEVICE
Frame 0: voxel= cuda:0 imu= cuda:0
Frame 1: voxel= cuda:0 imu= cuda:0
Frame 2: voxel= cuda:0 imu= cuda:0
Frame 3: voxel= cuda:0 imu= cuda:0

VOXELS
Shape : (2, 4, 5, 480, 640)
dtype : torch.float32
device: cuda:0

INPUT FINITE CHECK
OK: voxels                                        min=-4.742724e+00 max=6.468853e+00 mean=1.904134e-03
OK: frame[0].voxel_grid                           min=-2.971232e+00 max=4.828935e+00 mean=1.667969e-03
OK: frame[0].imu_angular_velocity                 min=-5.006700e-02 max=4.594000e-02 mean=2.604412e-03
OK: frame[0].imu_linear_acceleration              min=-8.993897e+00 max=8.293953e+0

/tmp/ipykernel_117851/692124854.py:500: UserWarning: Anomaly Detection has been enabled. This mode will increase the runtime and should only be enabled for debugging.
  with torch.autograd.detect_anomaly():
/tmp/ipykernel_117851/692124854.py:565: UserWarning: Anomaly Detection has been enabled. This mode will increase the runtime and should only be enabled for debugging.
  with torch.autograd.detect_anomaly():
/tmp/ipykernel_117851/692124854.py:771: UserWarning: Anomaly Detection has been enabled. This mode will increase the runtime and should only be enabled for debugging.
  with torch.autograd.detect_anomaly():


jij
Forward pass completed.

MODEL OUTPUT FINITE CHECK
OK: outputs['event_features']                     min=-2.784646e-01 max=2.313203e+01 mean=3.738647e-01
OK: outputs['motion_embeddings']                  min=-7.846074e-01 max=8.715376e-01 mean=-1.502089e-02
OK: outputs['fused_features']                     min=-2.784646e-01 max=1.489270e+01 mean=1.597312e-01
OK: outputs['temporal_features']                  min=-7.856124e-01 max=7.345730e-01 mean=-5.418432e-04
OK: outputs['depth']                              min=4.139850e-01 max=2.771977e+00 mean=7.854400e-01
OK: outputs['depths']                             min=2.676156e-01 max=3.372630e+00 mean=7.838554e-01
OK: outputs['poses']                              min=-2.915805e-01 max=1.820363e-01 mean=-1.204440e-01
OK: outputs['pose']                               min=-2.823136e-01 max=1.817925e-01 mean=-1.195257e-01
OK: outputs['predicted_state']                    min=-2.784646e-01 max=8.269917e+00 mean=2.083820e-01
OK: outputs['ren

In [9]:
# ============================================================
# 100-BATCH TRAINING STABILITY DIAGNOSTIC
# ============================================================
#
# Purpose:
#   Determine whether NaN/Inf appears because of:
#       1. a particular training batch,
#       2. exploding gradients,
#       3. an optimizer update,
#       4. model parameters becoming non-finite.
#
# This does NOT silently skip bad batches.
# It stops at the first numerical failure.
#
# ============================================================

from pathlib import Path

import torch
from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
    VoxelizeEvents,
)

from src.models.world_model.model import WorldModel
from src.losses.total_loss import TotalLoss


# ============================================================
# CONFIGURATION
# ============================================================

NUM_DIAGNOSTIC_BATCHES = 100

BATCH_SIZE = 2

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-5

VOXEL_BINS = 5

HISTORY_OFFSETS = (-3, -2, -1, 0)

GRAD_CLIP_NORM = 1.0

DATASET_ROOT = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 90)
print("DEVICE")
print("=" * 90)

print("Device:", device)

if device.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2,
        ),
        "GB",
    )


# ============================================================
# DATASET
# ============================================================

print()
print("=" * 90)
print("CREATING DATASET")
print("=" * 90)

frame_dataset = EVIMO2Dataset(
    dataset_root=DATASET_ROOT,
    sensors=(
        "left_camera",
        "right_camera",
    ),
    split="train",
    load_depth=True,
    load_mask=True,
)

temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,
    history_offsets=HISTORY_OFFSETS,
)

print()
print("Frame samples   :", len(frame_dataset))
print("Temporal samples:", len(temporal_dataset))
print("Batch size      :", BATCH_SIZE)


# ============================================================
# DATALOADER
# ============================================================

loader = DataLoader(
    temporal_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=temporal_collate_fn,
    num_workers=0,
    pin_memory=False,
)

print()
print("DataLoader ready.")


# ============================================================
# TRANSFORM PIPELINE
# ============================================================

print()
print("=" * 90)
print("TRANSFORM PIPELINE")
print("=" * 90)

transform = Compose(
    [
        ToTensor(),
        NormalizeEventTime(),
        NormalizeIMU(),
        VoxelizeEvents(
            num_bins=VOXEL_BINS,
        ),
    ]
)

print("Voxel bins:", VOXEL_BINS)
print("History offsets:", HISTORY_OFFSETS)


# ============================================================
# MODEL
# ============================================================

print()
print("=" * 90)
print("CREATING MODEL")
print("=" * 90)

model = WorldModel().to(device)

model.train()

num_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Trainable parameters:",
    num_parameters,
)


# ============================================================
# LOSS
# ============================================================

print()
print("=" * 90)
print("CREATING TOTAL LOSS")
print("=" * 90)

loss_fn = TotalLoss(
    latent_weight=1.0,
    depth_smoothness_weight=1.0,
    pose_temporal_weight=1.0,
    depth_temporal_weight=1.0,
    dynamic_mask_weight=1.0,
)

print("TotalLoss ready.")


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print()
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Gradient clipping:", GRAD_CLIP_NORM)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def check_tensor(
    name,
    tensor,
):
    """
    Check tensor for NaN/Inf and print statistics.
    """

    if not torch.is_tensor(tensor):
        return True

    finite = torch.isfinite(tensor)

    if not finite.all():

        print()
        print("!" * 90)
        print("NON-FINITE TENSOR")
        print("!" * 90)

        print("Tensor:", name)

        print(
            "NaN count:",
            torch.isnan(tensor).sum().item(),
        )

        print(
            "Inf count:",
            torch.isinf(tensor).sum().item(),
        )

        return False

    return True


def check_outputs(outputs):
    """
    Recursively check model outputs.
    """

    for name, value in outputs.items():

        if torch.is_tensor(value):

            if not check_tensor(
                f"outputs['{name}']",
                value,
            ):
                return False

    return True


def check_gradients(model):
    """
    Check every parameter gradient.

    Returns:
        (success, max_grad_norm, bad_parameter)
    """

    max_norm = 0.0

    bad_parameter = None

    for name, parameter in model.named_parameters():

        if parameter.grad is None:
            continue

        grad = parameter.grad

        if not torch.isfinite(grad).all():

            bad_parameter = name

            return (
                False,
                float("inf"),
                bad_parameter,
            )

        norm = grad.norm().item()

        if norm > max_norm:
            max_norm = norm

    return (
        True,
        max_norm,
        bad_parameter,
    )


def check_parameters(model):
    """
    Verify that all model parameters remain finite.
    """

    for name, parameter in model.named_parameters():

        if not torch.isfinite(parameter).all():

            return False, name

    return True, None


# ============================================================
# TRAINING DIAGNOSTIC
# ============================================================

print()
print("=" * 90)
print("STARTING 100-BATCH DIAGNOSTIC")
print("=" * 90)

print()
print(
    "The diagnostic will stop immediately if "
    "NaN/Inf is detected."
)

print()


# ------------------------------------------------------------
# Running statistics
# ------------------------------------------------------------

loss_history = []

grad_history = []

dynamic_ratio_history = []

max_grad_seen = 0.0

successful_batches = 0


# ============================================================
# BATCH LOOP
# ============================================================

for batch_idx, raw_batch in enumerate(loader):

    if batch_idx >= NUM_DIAGNOSTIC_BATCHES:
        break

    print()
    print("-" * 90)

    print(
        f"BATCH {batch_idx + 1}/{NUM_DIAGNOSTIC_BATCHES}"
    )

    print("-" * 90)


    # ========================================================
    # TRANSFORM
    # ========================================================

    voxel_batch = transform(raw_batch)


    # ========================================================
    # MOVE ENTIRE TEMPORAL BATCH TO GPU
    # ========================================================

    voxel_batch = voxel_batch.to(device)


    # ========================================================
    # BUILD VOXEL TENSOR
    # ========================================================

    voxels = torch.stack(
        [
            frame.voxel_grid
            for frame in voxel_batch.frames
        ],
        dim=1,
    )


    # ========================================================
    # INPUT CHECK
    # ========================================================

    if not torch.isfinite(voxels).all():

        print()
        print("!" * 90)
        print("FAILURE: NON-FINITE INPUT")
        print("!" * 90)

        print("Batch:", batch_idx)

        raise RuntimeError(
            "Non-finite voxel input detected."
        )


    if voxels.device.type != device.type:

        raise RuntimeError(
            f"Voxel device mismatch: "
            f"{voxels.device} != {device}"
        )


    print(
        "Voxel shape:",
        tuple(voxels.shape),
    )

    print(
        "Voxel device:",
        voxels.device,
    )


    # ========================================================
    # ZERO GRADIENT
    # ========================================================

    optimizer.zero_grad(
        set_to_none=True
    )


    # ========================================================
    # FORWARD PASS
    # ========================================================

    print("Forward...", end=" ")

    outputs = model(
        voxels,
        voxel_batch,
    )

    print("done")


    # ========================================================
    # OUTPUT CHECK
    # ========================================================

    if not check_outputs(outputs):

        print()
        print("!" * 90)
        print("FAILURE: NON-FINITE MODEL OUTPUT")
        print("!" * 90)

        print("Batch:", batch_idx)

        raise RuntimeError(
            "Non-finite model output detected."
        )


    # ========================================================
    # TOTAL LOSS
    # ========================================================

    loss_dict = loss_fn(
        outputs=outputs,
    )

    total_loss = loss_dict["loss"]


    # ========================================================
    # LOSS CHECK
    # ========================================================

    if not torch.isfinite(total_loss):

        print()
        print("!" * 90)
        print("FAILURE: NON-FINITE LOSS")
        print("!" * 90)

        print("Batch:", batch_idx)

        print(
            "Total loss:",
            total_loss,
        )

        for key, value in loss_dict.items():

            if torch.is_tensor(value):

                print(
                    f"{key:30s}:",
                    value.detach().item()
                    if value.ndim == 0
                    else value,
                )

        raise RuntimeError(
            "Non-finite total loss detected."
        )


    # ========================================================
    # PRINT LOSS
    # ========================================================

    print(
        f"Loss: {total_loss.detach().item():.8f}"
    )


    # ========================================================
    # BACKWARD
    # ========================================================

    print("Backward...", end=" ")

    total_loss.backward()

    print("done")


    # ========================================================
    # CHECK GRADIENTS BEFORE CLIPPING
    # ========================================================

    (
        gradients_finite,
        grad_norm,
        bad_parameter,
    ) = check_gradients(model)


    if not gradients_finite:

        print()
        print("!" * 90)
        print("FAILURE: NON-FINITE GRADIENT")
        print("!" * 90)

        print("Batch:", batch_idx)

        print(
            "Bad parameter:",
            bad_parameter,
        )

        print(
            "Total loss:",
            total_loss.detach().item(),
        )

        raise RuntimeError(
            "Non-finite gradient detected."
        )


    max_grad_seen = max(
        max_grad_seen,
        grad_norm,
    )

    grad_history.append(
        grad_norm
    )


    print(
        f"Gradient norm: {grad_norm:.6e}"
    )


    # ========================================================
    # GRADIENT CLIPPING
    # ========================================================

    clipped_norm = (
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_NORM,
        )
    )


    print(
        f"Gradient norm before clipping: "
        f"{clipped_norm:.6e}"
    )


    # ========================================================
    # CHECK GRADIENTS AFTER CLIPPING
    # ========================================================

    (
        gradients_finite,
        grad_norm_after,
        bad_parameter,
    ) = check_gradients(model)


    if not gradients_finite:

        raise RuntimeError(
            "Non-finite gradient after clipping."
        )


    # ========================================================
    # OPTIMIZER STEP
    # ========================================================

    print("Optimizer step...", end=" ")

    optimizer.step()

    print("done")


    # ========================================================
    # PARAMETER CHECK
    # ========================================================

    parameters_finite, bad_parameter = (
        check_parameters(model)
    )


    if not parameters_finite:

        print()
        print("!" * 90)
        print("FAILURE: NON-FINITE PARAMETER")
        print("!" * 90)

        print("Batch:", batch_idx)

        print(
            "Bad parameter:",
            bad_parameter,
        )

        raise RuntimeError(
            "Model parameter became non-finite "
            "after optimizer.step()."
        )


    # ========================================================
    # SAVE STATISTICS
    # ========================================================

    loss_history.append(
        total_loss.detach().item()
    )


    if "dynamic_ratio" in loss_dict:

        dynamic_ratio = (
            loss_dict["dynamic_ratio"]
            .detach()
            .item()
        )

        dynamic_ratio_history.append(
            dynamic_ratio
        )

        print(
            f"Dynamic ratio: {dynamic_ratio:.6f}"
        )


    # ========================================================
    # SUCCESS
    # ========================================================

    successful_batches += 1

    print(
        f"✓ Batch {batch_idx + 1} passed"
    )


# ============================================================
# FINAL RESULT
# ============================================================

print()
print("=" * 90)
print("DIAGNOSTIC RESULT")
print("=" * 90)

print()
print(
    "Successful batches:",
    successful_batches,
)

print(
    "Maximum gradient norm:",
    f"{max_grad_seen:.6e}",
)


# ============================================================
# LOSS STATISTICS
# ============================================================

if loss_history:

    loss_tensor = torch.tensor(
        loss_history
    )

    print()
    print("LOSS STATISTICS")
    print("-" * 90)

    print(
        "Minimum:",
        f"{loss_tensor.min().item():.8f}",
    )

    print(
        "Maximum:",
        f"{loss_tensor.max().item():.8f}",
    )

    print(
        "Mean:",
        f"{loss_tensor.mean().item():.8f}",
    )

    print(
        "First:",
        f"{loss_history[0]:.8f}",
    )

    print(
        "Last:",
        f"{loss_history[-1]:.8f}",
    )


# ============================================================
# GRADIENT STATISTICS
# ============================================================

if grad_history:

    grad_tensor = torch.tensor(
        grad_history
    )

    print()
    print("GRADIENT STATISTICS")
    print("-" * 90)

    print(
        "Minimum:",
        f"{grad_tensor.min().item():.6e}",
    )

    print(
        "Maximum:",
        f"{grad_tensor.max().item():.6e}",
    )

    print(
        "Mean:",
        f"{grad_tensor.mean().item():.6e}",
    )


# ============================================================
# DYNAMIC MASK STATISTICS
# ============================================================

if dynamic_ratio_history:

    ratio_tensor = torch.tensor(
        dynamic_ratio_history
    )

    print()
    print("DYNAMIC MASK STATISTICS")
    print("-" * 90)

    print(
        "Minimum:",
        f"{ratio_tensor.min().item():.6f}",
    )

    print(
        "Maximum:",
        f"{ratio_tensor.max().item():.6f}",
    )

    print(
        "Mean:",
        f"{ratio_tensor.mean().item():.6f}",
    )


# ============================================================
# CUDA MEMORY
# ============================================================

if device.type == "cuda":

    print()
    print("=" * 90)
    print("CUDA MEMORY")
    print("=" * 90)

    print(
        "Allocated:",
        round(
            torch.cuda.memory_allocated()
            / (1024 ** 3),
            3,
        ),
        "GB",
    )

    print(
        "Reserved:",
        round(
            torch.cuda.memory_reserved()
            / (1024 ** 3),
            3,
        ),
        "GB",
    )


# ============================================================
# FINAL CHECK
# ============================================================

if successful_batches == NUM_DIAGNOSTIC_BATCHES:

    print()
    print("=" * 90)
    print(
        "✓ 100-BATCH TRAINING DIAGNOSTIC PASSED"
    )
    print("=" * 90)

    print()
    print(
        "No NaN/Inf was detected in:"
    )

    print(
        "  ✓ Inputs"
    )

    print(
        "  ✓ Model outputs"
    )

    print(
        "  ✓ Total loss"
    )

    print(
        "  ✓ Gradients"
    )

    print(
        "  ✓ Model parameters"
    )

    print(
        "  ✓ Optimizer updates"
    )

    print()
    print(
        "Gradient clipping was active."
    )

else:

    print()
    print(
        "Diagnostic stopped before reaching "
        f"{NUM_DIAGNOSTIC_BATCHES} batches."
    )

DEVICE
Device: cuda
GPU: NVIDIA GeForce RTX 4070 SUPER
GPU memory: 11.59 GB

CREATING DATASET
EVIMO2 Sequence Index
Sequences : 22
Frames    : 9352
Sensors   : left_camera, right_camera
Split     : train

Frame samples   : 9352
Temporal samples: 9286
Batch size      : 2

DataLoader ready.

TRANSFORM PIPELINE
Voxel bins: 5
History offsets: (-3, -2, -1, 0)

CREATING MODEL
Trainable parameters: 15073368

CREATING TOTAL LOSS
TotalLoss ready.

Optimizer: AdamW
Learning rate: 0.0001
Weight decay: 1e-05
Gradient clipping: 1.0

STARTING 100-BATCH DIAGNOSTIC

The diagnostic will stop immediately if NaN/Inf is detected.


------------------------------------------------------------------------------------------
BATCH 1/100
------------------------------------------------------------------------------------------
Voxel shape: (2, 4, 5, 480, 640)
Voxel device: cuda:0
Forward... jij
done
Loss: 0.58802986
Backward... done
Gradient norm: 4.682581e+00
Gradient norm before clipping: 8.112626e+00
Optimi

In [2]:
# ============================================================
# EVIMO2 WORLD MODEL — 2 EPOCH GPU TRAINING
# ============================================================
#
# This cell:
#
#   1. Creates the EVIMO2 dataset
#   2. Creates the temporal dataset
#   3. Creates the DataLoader
#   4. Creates the preprocessing/voxelization pipeline
#   5. Creates the WorldModel
#   6. Creates TotalLoss
#   7. Creates the optimizer
#   8. Trains for 2 COMPLETE epochs
#   9. Uses gradient clipping
#  10. Checks inputs, loss, gradients and parameters for NaN/Inf
#  11. Reports gradient norms
#  12. Saves a checkpoint after every epoch
#
# ============================================================


from pathlib import Path
import math
import time

import torch
from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
    VoxelizeEvents,
)

from src.models.world_model.model import WorldModel

from src.losses.total_loss import TotalLoss


# ============================================================
# CONFIGURATION
# ============================================================

DATASET_ROOT = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

CHECKPOINT_DIR = Path(
    "/home/ayon/git/EventCameraProject/checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

NUM_EPOCHS = 2

BATCH_SIZE = 2

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-5

NUM_WORKERS = 0

PIN_MEMORY = False


# ------------------------------------------------------------
# Event representation
# ------------------------------------------------------------

NUM_BINS = 5


# ------------------------------------------------------------
# Temporal history
# ------------------------------------------------------------

HISTORY_OFFSETS = (
    -3,
    -2,
    -1,
    0,
)


# ------------------------------------------------------------
# Gradient clipping
# ------------------------------------------------------------

MAX_GRAD_NORM = 1.0


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 90)
print("TRAINING CONFIGURATION")
print("=" * 90)

print(f"Device          : {device}")


if device.type == "cuda":

    print(
        "GPU             :",
        torch.cuda.get_device_name(0),
    )

    total_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(
        f"GPU Memory      : {total_memory:.2f} GB"
    )


print(f"Epochs          : {NUM_EPOCHS}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Learning rate   : {LEARNING_RATE}")
print(f"Weight decay    : {WEIGHT_DECAY}")
print(f"Voxel bins      : {NUM_BINS}")
print(f"History offsets : {HISTORY_OFFSETS}")
print(f"Max grad norm   : {MAX_GRAD_NORM}")
print()


# ============================================================
# CUDA SETUP
# ============================================================

if device.type == "cuda":

    torch.backends.cudnn.benchmark = True

    torch.cuda.empty_cache()

    print(
        "CUDA available:",
        torch.cuda.is_available(),
    )

    print(
        "CUDA version:",
        torch.version.cuda,
    )

    print(
        "cuDNN version:",
        torch.backends.cudnn.version(),
    )

    print()


# ============================================================
# DATASET
# ============================================================

print("=" * 90)
print("CREATING DATASET")
print("=" * 90)


frame_dataset = EVIMO2Dataset(
    dataset_root=DATASET_ROOT,

    sensors=(
        "left_camera",
        "right_camera",
    ),

    split="train",

    load_depth=True,

    load_mask=True,
)


print(
    f"Frame samples   : {len(frame_dataset)}"
)


# ============================================================
# TEMPORAL DATASET
# ============================================================

temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,

    history_offsets=HISTORY_OFFSETS,
)


print(
    f"Temporal samples: {len(temporal_dataset)}"
)

print()


# ============================================================
# DATALOADER
# ============================================================

loader = DataLoader(
    temporal_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=PIN_MEMORY,

    collate_fn=temporal_collate_fn,

    drop_last=False,
)


print("DataLoader ready.")
print()


# ============================================================
# TRANSFORM PIPELINE
# ============================================================

print("=" * 90)
print("CREATING TRANSFORM PIPELINE")
print("=" * 90)


transform = Compose(
    [
        ToTensor(),

        NormalizeEventTime(),

        NormalizeIMU(),

        VoxelizeEvents(
            num_bins=NUM_BINS,
        ),
    ]
)


print(
    f"Voxel bins: {NUM_BINS}"
)

print()


# ============================================================
# MODEL
# ============================================================

print("=" * 90)
print("CREATING MODEL")
print("=" * 90)


model = WorldModel()

model = model.to(device)

model.train()


num_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)


print(
    f"Trainable parameters: {num_parameters:,}"
)

print()


# ============================================================
# LOSS
# ============================================================

print("=" * 90)
print("CREATING TOTAL LOSS")
print("=" * 90)


total_loss_fn = TotalLoss(

    latent_weight=1.0,

    depth_smoothness_weight=1.0,

    pose_temporal_weight=1.0,

    depth_temporal_weight=1.0,

    dynamic_mask_weight=1.0,
)


total_loss_fn = total_loss_fn.to(device)


print("TotalLoss ready.")
print()


# ============================================================
# OPTIMIZER
# ============================================================

print("=" * 90)
print("CREATING OPTIMIZER")
print("=" * 90)


optimizer = torch.optim.AdamW(
    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,
)


print("AdamW ready.")
print()


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def check_tensor_finite(
    tensor,
    name,
):
    """
    Raise an error if a tensor contains NaN or Inf.
    """

    if not torch.is_tensor(tensor):
        return

    if not torch.isfinite(tensor).all():

        print()
        print("=" * 90)
        print("NON-FINITE TENSOR DETECTED")
        print("=" * 90)

        print("Tensor:", name)

        print(
            "Shape:",
            tuple(tensor.shape),
        )

        raise RuntimeError(
            f"Non-finite tensor detected: {name}"
        )


def check_model_parameters():
    """
    Verify that all model parameters remain finite.
    """

    for name, parameter in model.named_parameters():

        if not torch.isfinite(
            parameter
        ).all():

            raise RuntimeError(
                "Non-finite model parameter detected: "
                + name
            )


def build_voxel_tensor(
    voxel_batch,
):
    """
    Construct the model event tensor from the
    CUDA-resident VoxelTemporalBatch.

    Shape:

        (B, T, C, H, W)

    where:

        B = batch size
        T = temporal sequence length
        C = voxel bins
    """

    voxels = torch.stack(
        [
            frame.voxel_grid
            for frame in voxel_batch.frames
        ],

        dim=1,
    )

    return voxels


# ============================================================
# TRAINING START
# ============================================================

print("=" * 90)
print("TRAINING START")
print("=" * 90)
print()


training_start = time.time()


# ============================================================
# EPOCH LOOP
# ============================================================

for epoch in range(
    1,
    NUM_EPOCHS + 1,
):

    epoch_start = time.time()

    model.train()

    total_epoch_loss = 0.0

    num_batches = 0

    num_clipped_batches = 0

    maximum_gradient_norm = 0.0

    minimum_loss = float("inf")

    maximum_loss = float("-inf")


    # --------------------------------------------------------
    # Epoch header
    # --------------------------------------------------------

    print()
    print("=" * 90)

    print(
        f"EPOCH {epoch}/{NUM_EPOCHS}"
    )

    print("=" * 90)


    # ========================================================
    # BATCH LOOP
    # ========================================================

    for batch_index, raw_batch in enumerate(
        loader,
        start=1,
    ):

        # ----------------------------------------------------
        # Clear gradients
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # Transform raw batch
        # ----------------------------------------------------

        voxel_batch = transform(
            raw_batch
        )


        # ----------------------------------------------------
        # Move ENTIRE temporal batch to GPU
        # ----------------------------------------------------

        voxel_batch = voxel_batch.to(
            device
        )


        # ----------------------------------------------------
        # Build voxel tensor AFTER .to(device)
        #
        # Important:
        #
        # Do NOT reuse a voxels tensor constructed before
        # voxel_batch.to(device).
        # ----------------------------------------------------

        voxels = build_voxel_tensor(
            voxel_batch
        )


        # ----------------------------------------------------
        # Input validation
        # ----------------------------------------------------

        assert voxels.ndim == 5

        assert (
            voxels.device.type == device.type
        ), (
            f"Voxel device mismatch: "
            f"{voxels.device} != {device}"
        )


        check_tensor_finite(
            voxels,
            "voxels",
        )


        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        outputs = model(
            voxels,
            voxel_batch,
        )


        # ----------------------------------------------------
        # Check model outputs
        # ----------------------------------------------------

        for name, value in outputs.items():

            if torch.is_tensor(value):

                check_tensor_finite(
                    value,
                    f"outputs['{name}']",
                )


        # ====================================================
        # TOTAL LOSS
        # ====================================================

        loss_dict = total_loss_fn(
            outputs=outputs
        )


        # ----------------------------------------------------
        # Check every loss component
        # ----------------------------------------------------

        for name, value in loss_dict.items():

            if torch.is_tensor(value):

                check_tensor_finite(
                    value,
                    f"loss['{name}']",
                )


        # ----------------------------------------------------
        # Main loss
        # ----------------------------------------------------

        loss = loss_dict["loss"]


        assert loss.ndim == 0


        if not torch.isfinite(loss):

            raise RuntimeError(
                f"Non-finite loss at "
                f"epoch={epoch}, "
                f"batch={batch_index}"
            )


        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        loss.backward()


        # ====================================================
        # GRADIENT CHECK BEFORE CLIPPING
        # ====================================================

        raw_gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),

            max_norm=MAX_GRAD_NORM,
        )


        # ----------------------------------------------------
        # clip_grad_norm_ returns the norm BEFORE clipping
        # ----------------------------------------------------

        if not torch.isfinite(
            raw_gradient_norm
        ):

            raise RuntimeError(
                "Non-finite gradient norm detected at "
                f"epoch={epoch}, "
                f"batch={batch_index}"
            )


        raw_gradient_norm_value = (
            raw_gradient_norm.item()
        )


        maximum_gradient_norm = max(
            maximum_gradient_norm,

            raw_gradient_norm_value,
        )


        if (
            raw_gradient_norm_value
            > MAX_GRAD_NORM
        ):

            num_clipped_batches += 1


        # ====================================================
        # INDIVIDUAL GRADIENT VALIDATION
        # ====================================================

        for name, parameter in model.named_parameters():

            if parameter.grad is None:
                continue


            if not torch.isfinite(
                parameter.grad
            ).all():

                print()
                print("=" * 90)

                print(
                    "NON-FINITE GRADIENT"
                )

                print("=" * 90)

                print(
                    "Epoch :",
                    epoch,
                )

                print(
                    "Batch :",
                    batch_index,
                )

                print(
                    "Parameter:",
                    name,
                )

                print(
                    "Gradient norm:",
                    parameter.grad.norm().item(),
                )

                raise RuntimeError(
                    "Non-finite gradient detected."
                )


        # ====================================================
        # OPTIMIZER STEP
        # ====================================================

        optimizer.step()


        # ====================================================
        # PARAMETER VALIDATION
        # ====================================================

        check_model_parameters()


        # ====================================================
        # STATISTICS
        # ====================================================

        loss_value = loss.item()


        total_epoch_loss += loss_value

        num_batches += 1


        minimum_loss = min(
            minimum_loss,
            loss_value,
        )


        maximum_loss = max(
            maximum_loss,
            loss_value,
        )


        # ====================================================
        # PROGRESS PRINT
        # ====================================================

        if (
            batch_index == 1
            or batch_index % 100 == 0
            or batch_index == len(loader)
        ):

            progress = (
                100.0
                * batch_index
                / len(loader)
            )


            print(
                f"[Epoch {epoch}/{NUM_EPOCHS}] "
                f"Batch {batch_index:5d}/{len(loader):5d} "
                f"({progress:6.2f}%) "
                f"Loss: {loss_value:.6f} "
                f"GradNorm: {raw_gradient_norm_value:.6f}"
            )


    # ========================================================
    # EPOCH STATISTICS
    # ========================================================

    average_epoch_loss = (
        total_epoch_loss
        / num_batches
    )


    epoch_time = (
        time.time()
        - epoch_start
    )


    print()
    print("-" * 90)

    print(
        f"EPOCH {epoch} COMPLETE"
    )

    print("-" * 90)

    print(
        f"Batches              : {num_batches}"
    )

    print(
        f"Average loss         : "
        f"{average_epoch_loss:.6f}"
    )

    print(
        f"Minimum loss         : "
        f"{minimum_loss:.6f}"
    )

    print(
        f"Maximum loss         : "
        f"{maximum_loss:.6f}"
    )

    print(
        f"Maximum grad norm    : "
        f"{maximum_gradient_norm:.6f}"
    )

    print(
        f"Clipped batches      : "
        f"{num_clipped_batches}"
    )

    print(
        f"Epoch time           : "
        f"{epoch_time / 60:.2f} min"
    )


    # ========================================================
    # CUDA MEMORY
    # ========================================================

    if device.type == "cuda":

        allocated = (
            torch.cuda.memory_allocated()
            / (1024 ** 3)
        )

        reserved = (
            torch.cuda.memory_reserved()
            / (1024 ** 3)
        )

        print()

        print(
            f"CUDA allocated       : "
            f"{allocated:.3f} GB"
        )

        print(
            f"CUDA reserved        : "
            f"{reserved:.3f} GB"
        )


    # ========================================================
    # SAVE CHECKPOINT
    # ========================================================

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"world_model_epoch_{epoch}.pt"
    )


    torch.save(
        {
            "epoch": epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "average_loss":
                average_epoch_loss,

            "learning_rate":
                LEARNING_RATE,

            "weight_decay":
                WEIGHT_DECAY,

            "batch_size":
                BATCH_SIZE,

            "voxel_bins":
                NUM_BINS,

            "history_offsets":
                HISTORY_OFFSETS,

            "max_grad_norm":
                MAX_GRAD_NORM,
        },

        checkpoint_path,
    )


    print()

    print(
        "Checkpoint saved:",
        checkpoint_path,
    )


# ============================================================
# TRAINING COMPLETE
# ============================================================

total_training_time = (
    time.time()
    - training_start
)


print()
print("=" * 90)

print(
    "✓ TRAINING COMPLETE"
)

print("=" * 90)

print()

print(
    f"Epochs completed : {NUM_EPOCHS}"
)

print(
    f"Total time       : "
    f"{total_training_time / 60:.2f} min"
)

print()

print(
    "Final model parameters are finite."
)

print(
    "Gradient clipping was active."
)

print(
    "All optimizer updates passed finite-value checks."
)

print()

print("=" * 90)
print("CHECKPOINTS")
print("=" * 90)

for epoch in range(
    1,
    NUM_EPOCHS + 1,
):

    path = (
        CHECKPOINT_DIR
        / f"world_model_epoch_{epoch}.pt"
    )

    print(
        path
    )

print()

print(
    "✓ 2-EPOCH TRAINING PASSED"
)

print("=" * 90)

TRAINING CONFIGURATION
Device          : cuda
GPU             : NVIDIA GeForce RTX 4070 SUPER
GPU Memory      : 11.59 GB
Epochs          : 2
Batch size      : 2
Learning rate   : 0.0001
Weight decay    : 1e-05
Voxel bins      : 5
History offsets : (-3, -2, -1, 0)
Max grad norm   : 1.0

CUDA available: True
CUDA version: 13.0
cuDNN version: 92000

CREATING DATASET
EVIMO2 Sequence Index
Sequences : 22
Frames    : 9352
Sensors   : left_camera, right_camera
Split     : train
Frame samples   : 9352
Temporal samples: 9286

DataLoader ready.

CREATING TRANSFORM PIPELINE
Voxel bins: 5

CREATING MODEL
Trainable parameters: 15,073,368

CREATING TOTAL LOSS
TotalLoss ready.

CREATING OPTIMIZER
AdamW ready.

TRAINING START


EPOCH 1/2
[Epoch 1/2] Batch     1/ 4643 (  0.02%) Loss: 0.680416 GradNorm: 20.241291
[Epoch 1/2] Batch   100/ 4643 (  2.15%) Loss: 0.157664 GradNorm: 0.388548
[Epoch 1/2] Batch   200/ 4643 (  4.31%) Loss: 0.142128 GradNorm: 0.339070
[Epoch 1/2] Batch   300/ 4643 (  6.46%) Loss:

In [4]:
# ============================================================
# EVIMO2 WORLD MODEL TRAINING
# Higher Batch Size / Faster Training
# ============================================================

from pathlib import Path
import time
import math

import torch
from torch.utils.data import DataLoader

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
    VoxelizeEvents,
)

from src.models.world_model.model import WorldModel
from src.losses.total_loss import TotalLoss


# ============================================================
# CONFIGURATION
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATASET_ROOT = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

EPOCHS = 2

# ------------------------------------------------------------
# Increased batch size
# ------------------------------------------------------------

BATCH_SIZE = 2

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5

VOXEL_BINS = 5

HISTORY_OFFSETS = (
    -3,
    -2,
    -1,
    0,
)

MAX_GRAD_NORM = 1.0

PRINT_EVERY = 100


# ============================================================
# CUDA CONFIGURATION
# ============================================================

if torch.cuda.is_available():

    torch.backends.cudnn.benchmark = True

    GPU_NAME = torch.cuda.get_device_name(0)

    GPU_MEMORY_GB = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

else:

    GPU_NAME = "CPU"
    GPU_MEMORY_GB = 0.0


# ============================================================
# CONFIGURATION PRINT
# ============================================================

print()
print("=" * 90)
print("TRAINING CONFIGURATION")
print("=" * 90)

print(f"Device          : {DEVICE}")
print(f"GPU             : {GPU_NAME}")
print(f"GPU Memory      : {GPU_MEMORY_GB:.2f} GB")

print(f"Epochs          : {EPOCHS}")
print(f"Batch size      : {BATCH_SIZE}")

print(f"Learning rate   : {LEARNING_RATE}")
print(f"Weight decay    : {WEIGHT_DECAY}")

print(f"Voxel bins      : {VOXEL_BINS}")
print(f"History offsets : {HISTORY_OFFSETS}")

print(f"Max grad norm   : {MAX_GRAD_NORM}")

if torch.cuda.is_available():

    print()
    print("CUDA available:", torch.cuda.is_available())
    print("CUDA version   :", torch.version.cuda)
    print(
        "cuDNN version  :",
        torch.backends.cudnn.version(),
    )


# ============================================================
# DATASET
# ============================================================

print()
print("=" * 90)
print("CREATING DATASET")
print("=" * 90)


frame_dataset = EVIMO2Dataset(

    dataset_root=DATASET_ROOT,

    sensors=(
        "left_camera",
        "right_camera",
    ),

    split="train",

    load_depth=True,

    load_mask=True,
)


temporal_dataset = TemporalEVIMO2Dataset(

    frame_dataset,

    history_offsets=HISTORY_OFFSETS,
)


print()
print("Frame samples   :", len(frame_dataset))
print("Temporal samples:", len(temporal_dataset))


# ============================================================
# DATALOADER
# ============================================================

loader = DataLoader(

    temporal_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    collate_fn=temporal_collate_fn,

    num_workers=0,

    pin_memory=torch.cuda.is_available(),

)


print()
print("DataLoader ready.")


# ============================================================
# TRANSFORM PIPELINE
# ============================================================

print()
print("=" * 90)
print("CREATING TRANSFORM PIPELINE")
print("=" * 90)


transform = Compose(
    [

        ToTensor(),

        NormalizeEventTime(),

        NormalizeIMU(),

        VoxelizeEvents(
            num_bins=VOXEL_BINS,
        ),

    ]
)


print(f"Voxel bins: {VOXEL_BINS}")


# ============================================================
# MODEL
# ============================================================

print()
print("=" * 90)
print("CREATING MODEL")
print("=" * 90)


model = WorldModel().to(DEVICE)

model.train()


trainable_parameters = sum(

    parameter.numel()

    for parameter in model.parameters()

    if parameter.requires_grad

)


print(
    f"Trainable parameters: {trainable_parameters:,}"
)


# ============================================================
# LOSS
# ============================================================

print()
print("=" * 90)
print("CREATING TOTAL LOSS")
print("=" * 90)


loss_fn = TotalLoss(

    latent_weight=1.0,

    depth_smoothness_weight=1.0,

    pose_temporal_weight=1.0,

    depth_temporal_weight=1.0,

    dynamic_mask_weight=1.0,

)


loss_fn = loss_fn.to(DEVICE)


print("TotalLoss ready.")


# ============================================================
# OPTIMIZER
# ============================================================

print()
print("=" * 90)
print("CREATING OPTIMIZER")
print("=" * 90)


optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,

)


print("AdamW ready.")


# ============================================================
# TRAINING
# ============================================================

print()
print("=" * 90)
print("TRAINING START")
print("=" * 90)


global_step = 0


for epoch in range(EPOCHS):

    model.train()

    epoch_start = time.time()

    epoch_losses = []

    epoch_grad_norms = []

    clipped_batches = 0

    total_batches = len(loader)


    print()
    print("=" * 90)
    print(
        f"EPOCH {epoch + 1}/{EPOCHS}"
    )
    print("=" * 90)


    for batch_idx, raw_batch in enumerate(loader):

        global_step += 1


        # ----------------------------------------------------
        # Transform
        # ----------------------------------------------------

        voxel_batch = transform(raw_batch)


        # ----------------------------------------------------
        # Move complete temporal batch to GPU
        # ----------------------------------------------------

        voxel_batch = voxel_batch.to(DEVICE)


        # ----------------------------------------------------
        # Build voxel tensor
        #
        # Shape:
        #
        # (B, T, C, H, W)
        #
        # ----------------------------------------------------

        voxels = torch.stack(

            [

                frame.voxel_grid

                for frame in voxel_batch.frames

            ],

            dim=1,

        )


        assert voxels.ndim == 5

        assert voxels.device.type == DEVICE.type


        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        outputs = model(

            voxels,

            voxel_batch,

        )


        # ----------------------------------------------------
        # Total loss
        # ----------------------------------------------------

        loss_output = loss_fn(

            outputs=outputs

        )


        total_loss = loss_output["loss"]


        # ----------------------------------------------------
        # Numerical safety
        # ----------------------------------------------------

        if not torch.isfinite(total_loss):

            print()
            print(
                "ERROR: Non-finite loss detected."
            )

            print(
                "Epoch:",
                epoch + 1,
            )

            print(
                "Batch:",
                batch_idx + 1,
            )

            raise RuntimeError(
                "Non-finite loss detected."
            )


        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        total_loss.backward()


        # ----------------------------------------------------
        # Gradient clipping
        # ----------------------------------------------------

        grad_norm = torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=MAX_GRAD_NORM,

        )


        grad_norm_value = float(
            grad_norm
        )


        epoch_grad_norms.append(
            grad_norm_value
        )


        if grad_norm_value > MAX_GRAD_NORM:

            clipped_batches += 1


        # ----------------------------------------------------
        # Check gradients
        # ----------------------------------------------------

        for name, parameter in model.named_parameters():

            if parameter.grad is None:

                continue


            if not torch.isfinite(
                parameter.grad
            ).all():

                print()
                print(
                    "ERROR: Non-finite gradient:"
                )

                print(name)

                raise RuntimeError(
                    "Non-finite gradient detected."
                )


        # ----------------------------------------------------
        # Optimizer step
        # ----------------------------------------------------

        optimizer.step()


        # ----------------------------------------------------
        # Check parameters
        # ----------------------------------------------------

        for name, parameter in model.named_parameters():

            if not torch.isfinite(
                parameter
            ).all():

                print()
                print(
                    "ERROR: Non-finite parameter:"
                )

                print(name)

                raise RuntimeError(
                    "Non-finite parameter detected."
                )


        # ----------------------------------------------------
        # Record loss
        # ----------------------------------------------------

        loss_value = float(
            total_loss.detach().cpu()
        )

        epoch_losses.append(
            loss_value
        )


        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (

            (batch_idx + 1) % PRINT_EVERY == 0

            or batch_idx == 0

            or batch_idx + 1 == total_batches

        ):

            progress = (
                (batch_idx + 1)
                / total_batches
                * 100.0
            )


            print(

                f"[Epoch {epoch + 1}/{EPOCHS}] "

                f"Batch {batch_idx + 1:5d}/"
                f"{total_batches:5d} "

                f"({progress:6.2f}%) "

                f"Loss: {loss_value:.6f} "

                f"GradNorm: "
                f"{grad_norm_value:.6f}"

            )


    # ========================================================
    # EPOCH SUMMARY
    # ========================================================

    epoch_time = (
        time.time()
        - epoch_start
    )


    average_loss = (
        sum(epoch_losses)
        / len(epoch_losses)
    )


    minimum_loss = min(
        epoch_losses
    )


    maximum_loss = max(
        epoch_losses
    )


    maximum_grad_norm = max(
        epoch_grad_norms
    )


    print()
    print("-" * 90)

    print(
        f"EPOCH {epoch + 1} COMPLETE"
    )

    print("-" * 90)

    print(
        f"Batches              : "
        f"{total_batches}"
    )

    print(
        f"Average loss         : "
        f"{average_loss:.6f}"
    )

    print(
        f"Minimum loss         : "
        f"{minimum_loss:.6f}"
    )

    print(
        f"Maximum loss         : "
        f"{maximum_loss:.6f}"
    )

    print(
        f"Maximum grad norm    : "
        f"{maximum_grad_norm:.6f}"
    )

    print(
        f"Clipped batches      : "
        f"{clipped_batches}"
    )

    print(
        f"Epoch time           : "
        f"{epoch_time / 60:.2f} min"
    )


    if torch.cuda.is_available():

        allocated = (
            torch.cuda.memory_allocated()
            / (1024 ** 3)
        )

        reserved = (
            torch.cuda.memory_reserved()
            / (1024 ** 3)
        )

        print()
        print(
            f"CUDA allocated       : "
            f"{allocated:.3f} GB"
        )

        print(
            f"CUDA reserved        : "
            f"{reserved:.3f} GB"
        )


# ============================================================
# TRAINING COMPLETE
# ============================================================

print()
print("=" * 90)
print("TRAINING COMPLETE")
print("=" * 90)

print(
    f"Completed {EPOCHS} epochs."
)

print(
    f"Batch size: {BATCH_SIZE}"
)

print(
    f"Total optimizer steps: "
    f"{EPOCHS * len(loader):,}"
)

if torch.cuda.is_available():

    print()
    print(
        f"Peak CUDA allocated: "
        f"{torch.cuda.max_memory_allocated() / (1024 ** 3):.3f} GB"
    )

    print(
        f"Peak CUDA reserved : "
        f"{torch.cuda.max_memory_reserved() / (1024 ** 3):.3f} GB"
    )

print()
print("=" * 90)


TRAINING CONFIGURATION
Device          : cuda
GPU             : NVIDIA GeForce RTX 4070 SUPER
GPU Memory      : 11.59 GB
Epochs          : 2
Batch size      : 2
Learning rate   : 0.0001
Weight decay    : 1e-05
Voxel bins      : 5
History offsets : (-3, -2, -1, 0)
Max grad norm   : 1.0

CUDA available: True
CUDA version   : 13.0
cuDNN version  : 92000

CREATING DATASET
EVIMO2 Sequence Index
Sequences : 22
Frames    : 9352
Sensors   : left_camera, right_camera
Split     : train

Frame samples   : 9352
Temporal samples: 9286

DataLoader ready.

CREATING TRANSFORM PIPELINE
Voxel bins: 5

CREATING MODEL
Trainable parameters: 15,073,368

CREATING TOTAL LOSS
TotalLoss ready.

CREATING OPTIMIZER
AdamW ready.

TRAINING START

EPOCH 1/2
[Epoch 1/2] Batch     1/ 4643 (  0.02%) Loss: 0.724952 GradNorm: 55.277641
[Epoch 1/2] Batch   100/ 4643 (  2.15%) Loss: 0.136088 GradNorm: 0.420314
[Epoch 1/2] Batch   200/ 4643 (  4.31%) Loss: 0.117343 GradNorm: 0.443078
[Epoch 1/2] Batch   300/ 4643 (  6.46%)

KeyboardInterrupt: 

In [6]:
# ============================================================
# EVIMO2 WORLD MODEL
# INDIVIDUAL LOSS MONITORING TRAINING
#
# Purpose:
#   Determine whether the model is learning meaningful
#   representations before optimizing training speed.
#
# Physical batch size:
#   2
#
# Reports every:
#   100 batches
#
# ============================================================

from pathlib import Path
import time
import math

import torch
from torch.utils.data import DataLoader


# ============================================================
# IMPORTS
# ============================================================

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
    VoxelizeEvents,
)

from src.models.world_model.model import WorldModel
from src.losses.total_loss import TotalLoss


# ============================================================
# CONFIGURATION
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

DATASET_ROOT = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

EPOCHS = 2

BATCH_SIZE = 2

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-5

MAX_GRAD_NORM = 1.0

# ------------------------------------------------------------
# Event representation
# ------------------------------------------------------------

VOXEL_BINS = 5

HISTORY_OFFSETS = (
    -3,
    -2,
    -1,
    0,
)

# ------------------------------------------------------------
# Monitoring
# ------------------------------------------------------------

PRINT_EVERY = 100


# ============================================================
# CUDA
# ============================================================

if torch.cuda.is_available():

    torch.backends.cudnn.benchmark = True

    GPU_NAME = torch.cuda.get_device_name(0)

    GPU_MEMORY_GB = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / (1024 ** 3)
    )

else:

    GPU_NAME = "CPU"

    GPU_MEMORY_GB = 0.0


# ============================================================
# CONFIGURATION
# ============================================================

print()
print("=" * 90)
print("TRAINING CONFIGURATION")
print("=" * 90)

print(f"Device          : {DEVICE}")
print(f"GPU             : {GPU_NAME}")
print(f"GPU Memory      : {GPU_MEMORY_GB:.2f} GB")

print(f"Epochs          : {EPOCHS}")
print(f"Batch size      : {BATCH_SIZE}")

print(f"Learning rate   : {LEARNING_RATE}")
print(f"Weight decay    : {WEIGHT_DECAY}")

print(f"Voxel bins      : {VOXEL_BINS}")
print(f"History offsets : {HISTORY_OFFSETS}")

print(f"Max grad norm   : {MAX_GRAD_NORM}")

print(f"Print every     : {PRINT_EVERY} batches")


# ============================================================
# DATASET
# ============================================================

print()
print("=" * 90)
print("CREATING DATASET")
print("=" * 90)


frame_dataset = EVIMO2Dataset(

    dataset_root=DATASET_ROOT,

    sensors=(
        "left_camera",
        "right_camera",
    ),

    split="train",

    load_depth=True,

    load_mask=True,

)


temporal_dataset = TemporalEVIMO2Dataset(

    frame_dataset,

    history_offsets=HISTORY_OFFSETS,

)


print()
print(
    f"Frame samples   : "
    f"{len(frame_dataset)}"
)

print(
    f"Temporal samples: "
    f"{len(temporal_dataset)}"
)


# ============================================================
# DATALOADER
# ============================================================

loader = DataLoader(

    temporal_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    collate_fn=temporal_collate_fn,

    num_workers=0,

    pin_memory=torch.cuda.is_available(),

)


print()
print("DataLoader ready.")


# ============================================================
# TRANSFORM PIPELINE
# ============================================================

print()
print("=" * 90)
print("CREATING TRANSFORM PIPELINE")
print("=" * 90)


transform = Compose(

    [

        ToTensor(),

        NormalizeEventTime(),

        NormalizeIMU(),

        VoxelizeEvents(
            num_bins=VOXEL_BINS
        ),

    ]

)


print(
    f"Voxel bins: {VOXEL_BINS}"
)


# ============================================================
# MODEL
# ============================================================

print()
print("=" * 90)
print("CREATING MODEL")
print("=" * 90)


model = WorldModel().to(DEVICE)

model.train()


trainable_parameters = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad

)


print(
    f"Trainable parameters: "
    f"{trainable_parameters:,}"
)


# ============================================================
# LOSS
# ============================================================

print()
print("=" * 90)
print("CREATING TOTAL LOSS")
print("=" * 90)


loss_fn = TotalLoss(

    latent_weight=1.0,

    depth_smoothness_weight=1.0,

    pose_temporal_weight=1.0,

    depth_temporal_weight=1.0,

    dynamic_mask_weight=1.0,

).to(DEVICE)


print("TotalLoss ready.")


# ============================================================
# OPTIMIZER
# ============================================================

print()
print("=" * 90)
print("CREATING OPTIMIZER")
print("=" * 90)


optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,

)


print("AdamW ready.")


# ============================================================
# LOSS HISTORY
# ============================================================
#
# Every printed checkpoint is stored here.
#
# This lets us inspect the individual objectives after training.
#
# ============================================================

history = {

    "epoch": [],

    "batch": [],

    "total_loss": [],

    "prediction_loss": [],

    "rendering_loss": [],

    "agreement_loss": [],

    "latent_loss": [],

    "depth_smoothness_loss": [],

    "depth_temporal_loss": [],

    "pose_temporal_loss": [],

    "dynamic_mask_loss": [],

    "mask_sparsity_loss": [],

    "mask_confidence_loss": [],

    "dynamic_ratio": [],

    "grad_norm": [],

}


# ============================================================
# HELPER
# ============================================================

def safe_float(value):

    """
    Convert tensor/scalar to Python float.
    """

    if torch.is_tensor(value):

        return float(
            value.detach()
            .cpu()
            .item()
        )

    return float(value)


# ============================================================
# TRAINING
# ============================================================

print()
print("=" * 90)
print("TRAINING START")
print("=" * 90)


global_step = 0


for epoch in range(EPOCHS):

    model.train()

    epoch_start = time.time()


    # --------------------------------------------------------
    # Running sums
    # --------------------------------------------------------

    running = {

        "total_loss": 0.0,

        "prediction_loss": 0.0,

        "rendering_loss": 0.0,

        "agreement_loss": 0.0,

        "latent_loss": 0.0,

        "depth_smoothness_loss": 0.0,

        "depth_temporal_loss": 0.0,

        "pose_temporal_loss": 0.0,

        "dynamic_mask_loss": 0.0,

        "mask_sparsity_loss": 0.0,

        "mask_confidence_loss": 0.0,

        "dynamic_ratio": 0.0,

        "grad_norm": 0.0,

    }


    epoch_losses = []


    total_batches = len(loader)

    clipped_batches = 0


    print()
    print("=" * 90)

    print(
        f"EPOCH {epoch + 1}/{EPOCHS}"
    )

    print("=" * 90)


    # ========================================================
    # BATCH LOOP
    # ========================================================

    for batch_idx, raw_batch in enumerate(loader):

        global_step += 1


        # ----------------------------------------------------
        # Transform
        # ----------------------------------------------------

        voxel_batch = transform(
            raw_batch
        )


        # ----------------------------------------------------
        # Move to GPU
        # ----------------------------------------------------

        voxel_batch = voxel_batch.to(
            DEVICE
        )


        # ----------------------------------------------------
        # Construct voxel tensor
        #
        # Expected:
        #
        # B x T x C x H x W
        #
        # ----------------------------------------------------

        voxels = torch.stack(

            [

                frame.voxel_grid

                for frame
                in voxel_batch.frames

            ],

            dim=1,

        )


        # ----------------------------------------------------
        # Sanity checks
        # ----------------------------------------------------

        assert voxels.ndim == 5

        assert torch.isfinite(
            voxels
        ).all()


        # ----------------------------------------------------
        # Zero gradients
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # ====================================================
        # FORWARD
        # ====================================================

        outputs = model(

            voxels,

            voxel_batch,

        )


        # ====================================================
        # LOSS
        # ====================================================

        loss_output = loss_fn(

            outputs=outputs

        )


        total_loss = loss_output[
            "loss"
        ]


        # ----------------------------------------------------
        # Numerical safety
        # ----------------------------------------------------

        if not torch.isfinite(
            total_loss
        ):

            raise RuntimeError(

                f"Non-finite total loss "
                f"at epoch={epoch + 1}, "
                f"batch={batch_idx + 1}"

            )


        # ====================================================
        # EXTRACT INDIVIDUAL LOSSES
        # ====================================================

        prediction_loss = (
            loss_output[
                "prediction_loss"
            ]
        )

        rendering_loss = (
            loss_output[
                "rendering_loss"
            ]
        )

        agreement_loss = (
            loss_output[
                "agreement_loss"
            ]
        )

        latent_loss = (
            loss_output[
                "latent_loss"
            ]
        )

        depth_smoothness_loss = (
            loss_output[
                "depth_smoothness_loss"
            ]
        )

        depth_temporal_loss = (
            loss_output[
                "depth_temporal_loss"
            ]
        )

        pose_temporal_loss = (
            loss_output[
                "pose_temporal_loss"
            ]
        )

        dynamic_mask_loss = (
            loss_output[
                "dynamic_mask_loss"
            ]
        )

        mask_sparsity_loss = (
            loss_output[
                "mask_sparsity_loss"
            ]
        )

        mask_confidence_loss = (
            loss_output[
                "mask_confidence_loss"
            ]
        )

        dynamic_ratio = (
            loss_output[
                "dynamic_ratio"
            ]
        )


        # ====================================================
        # BACKWARD
        # ====================================================

        total_loss.backward()


        # ====================================================
        # GRADIENT CLIPPING
        # ====================================================

        grad_norm = (
            torch.nn.utils
            .clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM,
            )
        )


        grad_norm_value = safe_float(
            grad_norm
        )


        if (
            grad_norm_value
            > MAX_GRAD_NORM
        ):

            clipped_batches += 1


        # ====================================================
        # GRADIENT VALIDATION
        # ====================================================

        for name, parameter in (
            model.named_parameters()
        ):

            if parameter.grad is None:

                continue


            if not torch.isfinite(
                parameter.grad
            ).all():

                raise RuntimeError(

                    "Non-finite gradient "
                    f"detected: {name}"

                )


        # ====================================================
        # OPTIMIZER
        # ====================================================

        optimizer.step()


        # ====================================================
        # PARAMETER VALIDATION
        # ====================================================

        for name, parameter in (
            model.named_parameters()
        ):

            if not torch.isfinite(
                parameter
            ).all():

                raise RuntimeError(

                    "Non-finite parameter "
                    f"detected: {name}"

                )


        # ====================================================
        # CONVERT TO FLOAT
        # ====================================================

        values = {

            "total_loss":
                safe_float(total_loss),

            "prediction_loss":
                safe_float(prediction_loss),

            "rendering_loss":
                safe_float(rendering_loss),

            "agreement_loss":
                safe_float(agreement_loss),

            "latent_loss":
                safe_float(latent_loss),

            "depth_smoothness_loss":
                safe_float(
                    depth_smoothness_loss
                ),

            "depth_temporal_loss":
                safe_float(
                    depth_temporal_loss
                ),

            "pose_temporal_loss":
                safe_float(
                    pose_temporal_loss
                ),

            "dynamic_mask_loss":
                safe_float(
                    dynamic_mask_loss
                ),

            "mask_sparsity_loss":
                safe_float(
                    mask_sparsity_loss
                ),

            "mask_confidence_loss":
                safe_float(
                    mask_confidence_loss
                ),

            "dynamic_ratio":
                safe_float(
                    dynamic_ratio
                ),

            "grad_norm":
                grad_norm_value,

        }


        # ====================================================
        # UPDATE RUNNING SUMS
        # ====================================================

        for key in running:

            running[key] += values[key]


        epoch_losses.append(
            values["total_loss"]
        )


        # ====================================================
        # PRINT EVERY 100 BATCHES
        # ====================================================

        should_print = (

            (batch_idx + 1)
            % PRINT_EVERY == 0

            or batch_idx == 0

            or batch_idx + 1
            == total_batches

        )


        if should_print:

            count = batch_idx + 1


            # ------------------------------------------------
            # Running averages
            # ------------------------------------------------

            avg = {

                key:
                    running[key] / count

                for key in running

            }


            # ------------------------------------------------
            # Store checkpoint
            # ------------------------------------------------

            history["epoch"].append(
                epoch + 1
            )

            history["batch"].append(
                batch_idx + 1
            )

            for key in values:

                history[key].append(
                    values[key]
                )


            # ------------------------------------------------
            # Progress
            # ------------------------------------------------

            progress = (

                count
                / total_batches
                * 100.0

            )


            print()
            print("-" * 90)

            print(

                f"[Epoch {epoch + 1}/{EPOCHS}] "
                f"Batch {count}/{total_batches} "
                f"({progress:.2f}%)"

            )

            print("-" * 90)


            # ------------------------------------------------
            # Current batch
            # ------------------------------------------------

            print(
                "CURRENT BATCH"
            )

            print(
                f"Total Loss          : "
                f"{values['total_loss']:.6f}"
            )

            print(
                f"Prediction Loss     : "
                f"{values['prediction_loss']:.6f}"
            )

            print(
                f"Rendering Loss      : "
                f"{values['rendering_loss']:.6f}"
            )

            print(
                f"Agreement Loss      : "
                f"{values['agreement_loss']:.6f}"
            )

            print(
                f"Depth Smoothness    : "
                f"{values['depth_smoothness_loss']:.6f}"
            )

            print(
                f"Depth Temporal      : "
                f"{values['depth_temporal_loss']:.6f}"
            )

            print(
                f"Pose Temporal       : "
                f"{values['pose_temporal_loss']:.6f}"
            )

            print(
                f"Dynamic Mask Loss   : "
                f"{values['dynamic_mask_loss']:.6f}"
            )

            print(
                f"  Mask Sparsity     : "
                f"{values['mask_sparsity_loss']:.6f}"
            )

            print(
                f"  Mask Confidence   : "
                f"{values['mask_confidence_loss']:.6f}"
            )

            print(
                f"Dynamic Ratio       : "
                f"{values['dynamic_ratio']:.6f}"
            )

            print(
                f"Gradient Norm       : "
                f"{values['grad_norm']:.6f}"
            )


            # ------------------------------------------------
            # Running average
            # ------------------------------------------------

            print()
            print(
                "RUNNING AVERAGE"
            )

            print(
                f"Total Loss          : "
                f"{avg['total_loss']:.6f}"
            )

            print(
                f"Prediction Loss     : "
                f"{avg['prediction_loss']:.6f}"
            )

            print(
                f"Rendering Loss      : "
                f"{avg['rendering_loss']:.6f}"
            )

            print(
                f"Agreement Loss      : "
                f"{avg['agreement_loss']:.6f}"
            )

            print(
                f"Depth Smoothness    : "
                f"{avg['depth_smoothness_loss']:.6f}"
            )

            print(
                f"Depth Temporal      : "
                f"{avg['depth_temporal_loss']:.6f}"
            )

            print(
                f"Pose Temporal       : "
                f"{avg['pose_temporal_loss']:.6f}"
            )

            print(
                f"Dynamic Mask Loss   : "
                f"{avg['dynamic_mask_loss']:.6f}"
            )

            print(
                f"Mask Sparsity       : "
                f"{avg['mask_sparsity_loss']:.6f}"
            )

            print(
                f"Mask Confidence     : "
                f"{avg['mask_confidence_loss']:.6f}"
            )

            print(
                f"Dynamic Ratio       : "
                f"{avg['dynamic_ratio']:.6f}"
            )

            print(
                f"Gradient Norm       : "
                f"{avg['grad_norm']:.6f}"
            )


    # ========================================================
    # EPOCH SUMMARY
    # ========================================================

    epoch_time = (
        time.time()
        - epoch_start
    )


    print()
    print("=" * 90)

    print(
        f"EPOCH {epoch + 1} COMPLETE"
    )

    print("=" * 90)


    print(
        f"Batches              : "
        f"{total_batches}"
    )

    print(
        f"Average Total Loss   : "
        f"{sum(epoch_losses) / len(epoch_losses):.6f}"
    )

    print(
        f"Minimum Total Loss   : "
        f"{min(epoch_losses):.6f}"
    )

    print(
        f"Maximum Total Loss   : "
        f"{max(epoch_losses):.6f}"
    )

    print(
        f"Clipped batches      : "
        f"{clipped_batches}"
    )

    print(
        f"Epoch time           : "
        f"{epoch_time / 60:.2f} min"
    )


    if torch.cuda.is_available():

        print()

        print(
            f"CUDA allocated       : "
            f"{torch.cuda.memory_allocated() / 1024**3:.3f} GB"
        )

        print(
            f"CUDA reserved        : "
            f"{torch.cuda.memory_reserved() / 1024**3:.3f} GB"
        )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 90)
print("TRAINING COMPLETE")
print("=" * 90)

print(
    f"Epochs completed: {EPOCHS}"
)

print(
    f"Batch size      : {BATCH_SIZE}"
)

print(
    f"Total batches   : "
    f"{EPOCHS * len(loader):,}"
)

print()
print(
    "Loss history checkpoints:"
)

print(
    len(history["total_loss"])
)

print()
print("=" * 90)


TRAINING CONFIGURATION
Device          : cuda
GPU             : NVIDIA GeForce RTX 4070 SUPER
GPU Memory      : 11.59 GB
Epochs          : 2
Batch size      : 2
Learning rate   : 0.0001
Weight decay    : 1e-05
Voxel bins      : 5
History offsets : (-3, -2, -1, 0)
Max grad norm   : 1.0
Print every     : 100 batches

CREATING DATASET
EVIMO2 Sequence Index
Sequences : 22
Frames    : 9352
Sensors   : left_camera, right_camera
Split     : train

Frame samples   : 9352
Temporal samples: 9286

DataLoader ready.

CREATING TRANSFORM PIPELINE
Voxel bins: 5

CREATING MODEL
Trainable parameters: 15,073,368

CREATING TOTAL LOSS
TotalLoss ready.

CREATING OPTIMIZER
AdamW ready.

TRAINING START

EPOCH 1/2

------------------------------------------------------------------------------------------
[Epoch 1/2] Batch 1/4643 (0.02%)
------------------------------------------------------------------------------------------
CURRENT BATCH
Total Loss          : 0.775988
Prediction Loss     : 0.158928
Renderi

In [ ]:
# ============================================================
# EVIMO2 WORLD MODEL
# INDIVIDUAL LOSS MONITORING TRAINING
#
# Purpose:
# Determine whether the model is learning meaningful
# representations before optimizing training speed.
#
# IMPORTANT:
# Individual loss modules return UNWEIGHTED losses.
# TotalLoss applies all experiment-level weights.
#
# For this diagnostic experiment:
#     ALL LOSS WEIGHTS = 1.0
#
# Reports every:
#     100 batches
#
# Physical batch size:
#     2
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

from pathlib import Path
import time

import torch
from torch.utils.data import DataLoader


# ============================================================
# PROJECT IMPORTS
# ============================================================

from src.data.dataset import EVIMO2Dataset
from src.data.temporal_dataset import TemporalEVIMO2Dataset
from src.data.collate import temporal_collate_fn

from src.data.transforms import (
    Compose,
    ToTensor,
    NormalizeEventTime,
    NormalizeIMU,
    VoxelizeEvents,
)

from src.models.world_model.model import WorldModel
from src.losses.total_loss import TotalLoss


# ============================================================
# CONFIGURATION
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

DATASET_ROOT = Path(
    "/home/ayon/HDD/EventDatasets/EVIMO2_official"
)


# ============================================================
# TRAINING CONFIGURATION
# ============================================================

EPOCHS = 2

BATCH_SIZE = 2

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-5

MAX_GRAD_NORM = 1.0


# ============================================================
# EVENT REPRESENTATION
# ============================================================

VOXEL_BINS = 5

HISTORY_OFFSETS = (
    -3,
    -2,
    -1,
    0,
)


# ============================================================
# MONITORING
# ============================================================

PRINT_EVERY = 100


# ============================================================
# LOSS WEIGHTS
#
# IMPORTANT:
# These are EXPERIMENT-LEVEL weights.
#
# Individual loss modules themselves return raw/unweighted
# losses.
#
# Start with everything equal to 1.0 so that we can first
# diagnose the natural scale and behavior of every objective.
# ============================================================

PREDICTION_LOSS_WEIGHT = 1.0

RENDERING_LOSS_WEIGHT = 1.0

AGREEMENT_LOSS_WEIGHT = 1.0

DEPTH_SMOOTHNESS_WEIGHT = 1.0

POSE_TEMPORAL_WEIGHT = 1.0

DEPTH_TEMPORAL_WEIGHT = 1.0

SPARSITY_LOSS_WEIGHT = 1.0

CONFIDENCE_LOSS_WEIGHT = 1.0

DYNAMIC_RATIO_WEIGHT = 0.0


# ============================================================
# CUDA
# ============================================================

if torch.cuda.is_available():

    torch.backends.cudnn.benchmark = True

    GPU_NAME = torch.cuda.get_device_name(0)

    GPU_MEMORY_GB = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / (1024 ** 3)
    )

else:

    GPU_NAME = "CPU"

    GPU_MEMORY_GB = 0.0


# ============================================================
# TRAINING CONFIGURATION DISPLAY
# ============================================================

print()
print("=" * 90)
print("TRAINING CONFIGURATION")
print("=" * 90)

print(f"Device          : {DEVICE}")
print(f"GPU             : {GPU_NAME}")
print(f"GPU Memory      : {GPU_MEMORY_GB:.2f} GB")

print(f"Epochs          : {EPOCHS}")
print(f"Batch size      : {BATCH_SIZE}")

print(f"Learning rate   : {LEARNING_RATE}")
print(f"Weight decay    : {WEIGHT_DECAY}")

print(f"Voxel bins      : {VOXEL_BINS}")
print(f"History offsets : {HISTORY_OFFSETS}")

print(f"Max grad norm   : {MAX_GRAD_NORM}")

print(f"Print every     : {PRINT_EVERY} batches")


# ============================================================
# LOSS WEIGHT DISPLAY
# ============================================================

print()
print("=" * 90)
print("LOSS WEIGHTS")
print("=" * 90)

print(
    f"Prediction loss weight   : "
    f"{PREDICTION_LOSS_WEIGHT}"
)

print(
    f"Rendering loss weight    : "
    f"{RENDERING_LOSS_WEIGHT}"
)

print(
    f"Agreement loss weight    : "
    f"{AGREEMENT_LOSS_WEIGHT}"
)

print(
    f"Depth smoothness weight  : "
    f"{DEPTH_SMOOTHNESS_WEIGHT}"
)

print(
    f"Pose temporal weight     : "
    f"{POSE_TEMPORAL_WEIGHT}"
)

print(
    f"Depth temporal weight    : "
    f"{DEPTH_TEMPORAL_WEIGHT}"
)

print(
    f"Mask sparsity weight     : "
    f"{SPARSITY_LOSS_WEIGHT}"
)

print(
    f"Mask confidence weight   : "
    f"{CONFIDENCE_LOSS_WEIGHT}"
)

print(
    f"Dynamic ratio weight     : "
    f"{DYNAMIC_RATIO_WEIGHT}"
)


# ============================================================
# DATASET
# ============================================================

print()
print("=" * 90)
print("CREATING DATASET")
print("=" * 90)


frame_dataset = EVIMO2Dataset(
    dataset_root=DATASET_ROOT,

    sensors=(
        "left_camera",
        "right_camera",
    ),

    split="train",

    load_depth=True,

    load_mask=True,
)


temporal_dataset = TemporalEVIMO2Dataset(
    frame_dataset,

    history_offsets=HISTORY_OFFSETS,
)


print()
print(
    f"Frame samples   : "
    f"{len(frame_dataset)}"
)

print(
    f"Temporal samples: "
    f"{len(temporal_dataset)}"
)


# ============================================================
# DATALOADER
# ============================================================

loader = DataLoader(
    temporal_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    collate_fn=temporal_collate_fn,

    num_workers=0,

    pin_memory=torch.cuda.is_available(),
)


print()
print("DataLoader ready.")


# ============================================================
# TRANSFORM PIPELINE
# ============================================================

print()
print("=" * 90)
print("CREATING TRANSFORM PIPELINE")
print("=" * 90)


transform = Compose(
    [
        ToTensor(),

        NormalizeEventTime(),

        NormalizeIMU(),

        VoxelizeEvents(
            num_bins=VOXEL_BINS
        ),
    ]
)


print(
    f"Voxel bins: {VOXEL_BINS}"
)


# ============================================================
# MODEL
# ============================================================

print()
print("=" * 90)
print("CREATING MODEL")
print("=" * 90)


model = WorldModel().to(DEVICE)

model.train()


trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    f"Trainable parameters: "
    f"{trainable_parameters:,}"
)


# ============================================================
# TOTAL LOSS
#
# Individual losses are UNWEIGHTED.
# TotalLoss applies the experiment-level weights.
# ============================================================

print()
print("=" * 90)
print("CREATING TOTAL LOSS")
print("=" * 90)


loss_fn = TotalLoss(

    prediction_loss_weight=(
        PREDICTION_LOSS_WEIGHT
    ),

    rendering_loss_weight=(
        RENDERING_LOSS_WEIGHT
    ),

    agreement_loss_weight=(
        AGREEMENT_LOSS_WEIGHT
    ),

    depth_smoothness_weight=(
        DEPTH_SMOOTHNESS_WEIGHT
    ),

    pose_temporal_weight=(
        POSE_TEMPORAL_WEIGHT
    ),

    depth_temporal_weight=(
        DEPTH_TEMPORAL_WEIGHT
    ),

    sparsity_loss_weight=(
        SPARSITY_LOSS_WEIGHT
    ),

    confidence_loss_weight=(
        CONFIDENCE_LOSS_WEIGHT
    ),

    dynamic_ratio_weight=(
        DYNAMIC_RATIO_WEIGHT
    ),

).to(DEVICE)


print("TotalLoss ready.")


# ============================================================
# OPTIMIZER
# ============================================================

print()
print("=" * 90)
print("CREATING OPTIMIZER")
print("=" * 90)


optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,
)


print("AdamW ready.")


# ============================================================
# LOSS HISTORY
#
# Each checkpoint stores BOTH:
#
#   1. Raw/unweighted losses
#   2. Weighted contributions
#
# This is important for deciding which objective should
# eventually be suppressed or down-weighted.
# ============================================================

history = {

    "epoch": [],

    "batch": [],

    # --------------------------------------------------------
    # Total
    # --------------------------------------------------------

    "total_loss": [],

    # --------------------------------------------------------
    # Raw latent components
    # --------------------------------------------------------

    "prediction_loss": [],

    "rendering_loss": [],

    "agreement_loss": [],

    # --------------------------------------------------------
    # Weighted latent components
    # --------------------------------------------------------

    "weighted_prediction_loss": [],

    "weighted_rendering_loss": [],

    "weighted_agreement_loss": [],

    # --------------------------------------------------------
    # Other raw losses
    # --------------------------------------------------------

    "depth_smoothness_raw": [],

    "depth_temporal_raw": [],

    "pose_temporal_raw": [],

    "mask_sparsity_raw": [],

    "mask_confidence_raw": [],

    "dynamic_ratio_raw": [],

    # --------------------------------------------------------
    # Weighted contributions
    # --------------------------------------------------------

    "depth_smoothness_loss": [],

    "depth_temporal_loss": [],

    "pose_temporal_loss": [],

    "dynamic_mask_loss": [],

    # --------------------------------------------------------
    # Dynamic mask information
    # --------------------------------------------------------

    "dynamic_ratio": [],

    # --------------------------------------------------------
    # Optimization
    # --------------------------------------------------------

    "grad_norm": [],
}


# ============================================================
# HELPER
# ============================================================

def safe_float(value):
    """
    Convert tensor/scalar to Python float.
    """

    if torch.is_tensor(value):

        return float(
            value.detach()
            .cpu()
            .item()
        )

    return float(value)


# ============================================================
# TRAINING
# ============================================================

print()
print("=" * 90)
print("TRAINING START")
print("=" * 90)


global_step = 0


for epoch in range(EPOCHS):

    model.train()

    epoch_start = time.time()


    # ========================================================
    # RUNNING SUMS
    # ========================================================

    running = {

        "total_loss": 0.0,

        "prediction_loss": 0.0,

        "rendering_loss": 0.0,

        "agreement_loss": 0.0,

        "weighted_prediction_loss": 0.0,

        "weighted_rendering_loss": 0.0,

        "weighted_agreement_loss": 0.0,

        "depth_smoothness_raw": 0.0,

        "depth_temporal_raw": 0.0,

        "pose_temporal_raw": 0.0,

        "mask_sparsity_raw": 0.0,

        "mask_confidence_raw": 0.0,

        "dynamic_ratio_raw": 0.0,

        "depth_smoothness_loss": 0.0,

        "depth_temporal_loss": 0.0,

        "pose_temporal_loss": 0.0,

        "dynamic_mask_loss": 0.0,

        "dynamic_ratio": 0.0,

        "grad_norm": 0.0,
    }


    epoch_losses = []

    total_batches = len(loader)

    clipped_batches = 0


    print()
    print("=" * 90)

    print(
        f"EPOCH {epoch + 1}/{EPOCHS}"
    )

    print("=" * 90)


    # ========================================================
    # BATCH LOOP
    # ========================================================

    for batch_idx, raw_batch in enumerate(loader):

        global_step += 1


        # ====================================================
        # TRANSFORM
        # ====================================================

        voxel_batch = transform(
            raw_batch
        )


        # ====================================================
        # MOVE TO GPU
        # ====================================================

        voxel_batch = voxel_batch.to(
            DEVICE
        )


        # ====================================================
        # CONSTRUCT VOXEL TENSOR
        #
        # Expected:
        #
        # B x T x C x H x W
        # ====================================================

        voxels = torch.stack(

            [
                frame.voxel_grid

                for frame
                in voxel_batch.frames
            ],

            dim=1,
        )


        # ====================================================
        # INPUT SANITY CHECK
        # ====================================================

        assert voxels.ndim == 5

        if not torch.isfinite(
            voxels
        ).all():

            raise RuntimeError(
                "Non-finite voxel input detected."
            )


        # ====================================================
        # ZERO GRADIENT
        # ====================================================

        optimizer.zero_grad(
            set_to_none=True
        )


        # ====================================================
        # FORWARD
        # ====================================================

        outputs = model(

            voxels,

            voxel_batch,
        )


        # ====================================================
        # LOSS
        # ====================================================

        loss_output = loss_fn(
            outputs=outputs
        )


        total_loss = loss_output[
            "loss"
        ]


        # ====================================================
        # TOTAL LOSS SANITY CHECK
        # ====================================================

        if not torch.isfinite(
            total_loss
        ):

            raise RuntimeError(

                f"Non-finite total loss "
                f"at epoch={epoch + 1}, "
                f"batch={batch_idx + 1}"
            )


        # ====================================================
        # RAW LATENT LOSSES
        #
        # These are UNWEIGHTED.
        # ====================================================

        prediction_loss = (
            loss_output[
                "prediction_loss"
            ]
        )

        rendering_loss = (
            loss_output[
                "rendering_loss"
            ]
        )

        agreement_loss = (
            loss_output[
                "agreement_loss"
            ]
        )


        # ====================================================
        # WEIGHTED LATENT CONTRIBUTIONS
        #
        # TotalLoss applies these weights internally.
        #
        # We calculate these explicitly here only for
        # diagnostic reporting.
        # ====================================================

        weighted_prediction_loss = (

            PREDICTION_LOSS_WEIGHT
            * prediction_loss
        )

        weighted_rendering_loss = (

            RENDERING_LOSS_WEIGHT
            * rendering_loss
        )

        weighted_agreement_loss = (

            AGREEMENT_LOSS_WEIGHT
            * agreement_loss
        )


        # ====================================================
        # RAW OTHER LOSSES
        # ====================================================

        depth_smoothness_raw = (
            loss_output[
                "depth_smoothness_loss"
            ]
        )

        depth_temporal_raw = (
            loss_output[
                "depth_temporal_loss"
            ]
        )

        pose_temporal_raw = (
            loss_output[
                "pose_temporal_loss"
            ]
        )

        mask_sparsity_raw = (
            loss_output[
                "mask_sparsity_loss"
            ]
        )

        mask_confidence_raw = (
            loss_output[
                "mask_confidence_loss"
            ]
        )

        dynamic_ratio_raw = (
            loss_output[
                "dynamic_ratio"
            ]
        )


        # ====================================================
        # WEIGHTED CONTRIBUTIONS
        # ====================================================

        depth_smoothness_loss = (
            loss_output[
                "depth_smoothness_loss"
            ]
        )

        depth_temporal_loss = (
            loss_output[
                "depth_temporal_loss"
            ]
        )

        pose_temporal_loss = (
            loss_output[
                "pose_temporal_loss"
            ]
        )

        dynamic_mask_loss = (
            loss_output[
                "dynamic_mask_loss"
            ]
        )

        dynamic_ratio = (
            loss_output[
                "dynamic_ratio"
            ]
        )


        # ====================================================
        # BACKWARD
        # ====================================================

        total_loss.backward()


        # ====================================================
        # GRADIENT CLIPPING
        # ====================================================

        grad_norm = (
            torch.nn.utils
            .clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM,
            )
        )


        grad_norm_value = safe_float(
            grad_norm
        )


        if (
            grad_norm_value
            > MAX_GRAD_NORM
        ):

            clipped_batches += 1


        # ====================================================
        # GRADIENT VALIDATION
        # ====================================================

        for name, parameter in (
            model.named_parameters()
        ):

            if parameter.grad is None:
                continue


            if not torch.isfinite(
                parameter.grad
            ).all():

                raise RuntimeError(

                    "Non-finite gradient "
                    f"detected: {name}"
                )


        # ====================================================
        # OPTIMIZER
        # ====================================================

        optimizer.step()


        # ====================================================
        # PARAMETER VALIDATION
        # ====================================================

        for name, parameter in (
            model.named_parameters()
        ):

            if not torch.isfinite(
                parameter
            ).all():

                raise RuntimeError(

                    "Non-finite parameter "
                    f"detected: {name}"
                )


        # ====================================================
        # CONVERT TO PYTHON FLOATS
        # ====================================================

        values = {

            "total_loss":
                safe_float(total_loss),

            # Raw latent
            "prediction_loss":
                safe_float(
                    prediction_loss
                ),

            "rendering_loss":
                safe_float(
                    rendering_loss
                ),

            "agreement_loss":
                safe_float(
                    agreement_loss
                ),

            # Weighted latent
            "weighted_prediction_loss":
                safe_float(
                    weighted_prediction_loss
                ),

            "weighted_rendering_loss":
                safe_float(
                    weighted_rendering_loss
                ),

            "weighted_agreement_loss":
                safe_float(
                    weighted_agreement_loss
                ),

            # Raw other losses
            "depth_smoothness_raw":
                safe_float(
                    depth_smoothness_raw
                ),

            "depth_temporal_raw":
                safe_float(
                    depth_temporal_raw
                ),

            "pose_temporal_raw":
                safe_float(
                    pose_temporal_raw
                ),

            "mask_sparsity_raw":
                safe_float(
                    mask_sparsity_raw
                ),

            "mask_confidence_raw":
                safe_float(
                    mask_confidence_raw
                ),

            "dynamic_ratio_raw":
                safe_float(
                    dynamic_ratio_raw
                ),

            # Weighted contributions
            "depth_smoothness_loss":
                safe_float(
                    depth_smoothness_loss
                ),

            "depth_temporal_loss":
                safe_float(
                    depth_temporal_loss
                ),

            "pose_temporal_loss":
                safe_float(
                    pose_temporal_loss
                ),

            "dynamic_mask_loss":
                safe_float(
                    dynamic_mask_loss
                ),

            "dynamic_ratio":
                safe_float(
                    dynamic_ratio
                ),

            "grad_norm":
                grad_norm_value,
        }


        # ====================================================
        # UPDATE RUNNING SUMS
        # ====================================================

        for key in running:

            running[key] += values[key]


        epoch_losses.append(
            values["total_loss"]
        )


        # ====================================================
        # PRINT CHECKPOINT
        # ====================================================

        should_print = (

            (batch_idx + 1)
            % PRINT_EVERY == 0

            or batch_idx == 0

            or batch_idx + 1
            == total_batches
        )


        if should_print:

            count = batch_idx + 1


            # =================================================
            # RUNNING AVERAGES
            # =================================================

            avg = {

                key:
                    running[key] / count

                for key in running
            }


            # =================================================
            # STORE HISTORY
            # =================================================

            history["epoch"].append(
                epoch + 1
            )

            history["batch"].append(
                batch_idx + 1
            )

            for key in values:

                history[key].append(
                    values[key]
                )


            # =================================================
            # PROGRESS
            # =================================================

            progress = (
                count
                / total_batches
                * 100.0
            )


            print()
            print("-" * 90)

            print(

                f"[Epoch {epoch + 1}/{EPOCHS}] "
                f"Batch {count}/{total_batches} "
                f"({progress:.2f}%)"
            )

            print("-" * 90)


            # =================================================
            # CURRENT BATCH
            # =================================================

            print()
            print("CURRENT BATCH")

            print(
                f"Total Loss          : "
                f"{values['total_loss']:.6f}"
            )

            print()
            print("RAW LOSSES")

            print(
                f"Prediction Loss     : "
                f"{values['prediction_loss']:.6f}"
            )

            print(
                f"Rendering Loss      : "
                f"{values['rendering_loss']:.6f}"
            )

            print(
                f"Agreement Loss      : "
                f"{values['agreement_loss']:.6f}"
            )

            print(
                f"Depth Smoothness    : "
                f"{values['depth_smoothness_raw']:.6f}"
            )

            print(
                f"Depth Temporal      : "
                f"{values['depth_temporal_raw']:.6f}"
            )

            print(
                f"Pose Temporal       : "
                f"{values['pose_temporal_raw']:.6f}"
            )

            print(
                f"Mask Sparsity       : "
                f"{values['mask_sparsity_raw']:.6f}"
            )

            print(
                f"Mask Confidence     : "
                f"{values['mask_confidence_raw']:.6f}"
            )

            print(
                f"Dynamic Ratio       : "
                f"{values['dynamic_ratio_raw']:.6f}"
            )


            # =================================================
            # WEIGHTED CONTRIBUTIONS
            # =================================================

            print()
            print("WEIGHTED CONTRIBUTIONS")

            print(
                f"Prediction          : "
                f"{values['weighted_prediction_loss']:.6f}"
            )

            print(
                f"Rendering           : "
                f"{values['weighted_rendering_loss']:.6f}"
            )

            print(
                f"Agreement           : "
                f"{values['weighted_agreement_loss']:.6f}"
            )

            print(
                f"Depth Smoothness    : "
                f"{values['depth_smoothness_loss']:.6f}"
            )

            print(
                f"Depth Temporal      : "
                f"{values['depth_temporal_loss']:.6f}"
            )

            print(
                f"Pose Temporal       : "
                f"{values['pose_temporal_loss']:.6f}"
            )

            print(
                f"Dynamic Mask        : "
                f"{values['dynamic_mask_loss']:.6f}"
            )


            # =================================================
            # DYNAMIC MASK
            # =================================================

            print()
            print("DYNAMIC MASK")

            print(
                f"Dynamic Ratio       : "
                f"{values['dynamic_ratio']:.6f}"
            )


            # =================================================
            # OPTIMIZATION
            # =================================================

            print()
            print("OPTIMIZATION")

            print(
                f"Gradient Norm       : "
                f"{values['grad_norm']:.6f}"
            )


            # =================================================
            # RUNNING AVERAGE
            # =================================================

            print()
            print("RUNNING AVERAGE")

            print(
                f"Total Loss          : "
                f"{avg['total_loss']:.6f}"
            )

            print()
            print("Raw losses:")

            print(
                f"Prediction          : "
                f"{avg['prediction_loss']:.6f}"
            )

            print(
                f"Rendering           : "
                f"{avg['rendering_loss']:.6f}"
            )

            print(
                f"Agreement           : "
                f"{avg['agreement_loss']:.6f}"
            )

            print(
                f"Depth Smoothness    : "
                f"{avg['depth_smoothness_raw']:.6f}"
            )

            print(
                f"Depth Temporal      : "
                f"{avg['depth_temporal_raw']:.6f}"
            )

            print(
                f"Pose Temporal       : "
                f"{avg['pose_temporal_raw']:.6f}"
            )

            print(
                f"Mask Sparsity       : "
                f"{avg['mask_sparsity_raw']:.6f}"
            )

            print(
                f"Mask Confidence     : "
                f"{avg['mask_confidence_raw']:.6f}"
            )

            print()
            print("Weighted:")

            print(
                f"Prediction          : "
                f"{avg['weighted_prediction_loss']:.6f}"
            )

            print(
                f"Rendering           : "
                f"{avg['weighted_rendering_loss']:.6f}"
            )

            print(
                f"Agreement           : "
                f"{avg['weighted_agreement_loss']:.6f}"
            )

            print(
                f"Depth Smoothness    : "
                f"{avg['depth_smoothness_loss']:.6f}"
            )

            print(
                f"Depth Temporal      : "
                f"{avg['depth_temporal_loss']:.6f}"
            )

            print(
                f"Pose Temporal       : "
                f"{avg['pose_temporal_loss']:.6f}"
            )

            print(
                f"Dynamic Mask        : "
                f"{avg['dynamic_mask_loss']:.6f}"
            )

            print()
            print(
                f"Dynamic Ratio       : "
                f"{avg['dynamic_ratio']:.6f}"
            )

            print(
                f"Gradient Norm       : "
                f"{avg['grad_norm']:.6f}"
            )


    # ========================================================
    # EPOCH SUMMARY
    # ========================================================

    epoch_time = (
        time.time()
        - epoch_start
    )


    print()
    print("=" * 90)

    print(
        f"EPOCH {epoch + 1} COMPLETE"
    )

    print("=" * 90)


    print(
        f"Batches              : "
        f"{total_batches}"
    )

    print(
        f"Average Total Loss   : "
        f"{sum(epoch_losses) / len(epoch_losses):.6f}"
    )

    print(
        f"Minimum Total Loss   : "
        f"{min(epoch_losses):.6f}"
    )

    print(
        f"Maximum Total Loss   : "
        f"{max(epoch_losses):.6f}"
    )

    print(
        f"Clipped batches      : "
        f"{clipped_batches}"
    )

    print(
        f"Epoch time           : "
        f"{epoch_time / 60:.2f} min"
    )


    if torch.cuda.is_available():

        print()

        print(
            f"CUDA allocated       : "
            f"{torch.cuda.memory_allocated() / 1024**3:.3f} GB"
        )

        print(
            f"CUDA reserved        : "
            f"{torch.cuda.memory_reserved() / 1024**3:.3f} GB"
        )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 90)
print("TRAINING COMPLETE")
print("=" * 90)

print(
    f"Epochs completed: {EPOCHS}"
)

print(
    f"Batch size      : {BATCH_SIZE}"
)

print(
    f"Total batches   : "
    f"{EPOCHS * len(loader):,}"
)

print()

print(
    "Loss history checkpoints:"
)

print(
    len(history["total_loss"])
)

print()
print("=" * 90)


TRAINING CONFIGURATION
Device          : cuda
GPU             : NVIDIA GeForce RTX 4070 SUPER
GPU Memory      : 11.59 GB
Epochs          : 2
Batch size      : 2
Learning rate   : 0.0001
Weight decay    : 1e-05
Voxel bins      : 5
History offsets : (-3, -2, -1, 0)
Max grad norm   : 1.0
Print every     : 100 batches

LOSS WEIGHTS
Prediction loss weight   : 1.0
Rendering loss weight    : 1.0
Agreement loss weight    : 1.0
Depth smoothness weight  : 1.0
Pose temporal weight     : 1.0
Depth temporal weight    : 1.0
Mask sparsity weight     : 1.0
Mask confidence weight   : 1.0
Dynamic ratio weight     : 1.0

CREATING DATASET
EVIMO2 Sequence Index
Sequences : 22
Frames    : 9352
Sensors   : left_camera, right_camera
Split     : train

Frame samples   : 9352
Temporal samples: 9286

DataLoader ready.

CREATING TRANSFORM PIPELINE
Voxel bins: 5

CREATING MODEL
Trainable parameters: 15,081,560

CREATING TOTAL LOSS
TotalLoss ready.

CREATING OPTIMIZER
AdamW ready.

TRAINING START

EPOCH 1/2


ValueError: Using a target size (torch.Size([2, 1, 30, 40])) that is different to the input size (torch.Size([2, 1, 480, 640])) is deprecated. Please ensure they have the same size.

In [2]:
import torch
from collections import defaultdict

# ============================================================
# MODEL PARAMETER / ARCHITECTURE VERIFICATION
# Assumes:
#     model
# is already instantiated in the notebook.
# ============================================================

print("=" * 100)
print("MODEL PARAMETER & ARCHITECTURE VERIFICATION")
print("=" * 100)

# ------------------------------------------------------------
# 1. Basic model parameter statistics
# ------------------------------------------------------------

all_params = list(model.parameters())

total_params = sum(p.numel() for p in all_params)
trainable_params = sum(p.numel() for p in all_params if p.requires_grad)
frozen_params = total_params - trainable_params

print("\n[1] GLOBAL PARAMETER COUNT")
print("-" * 100)
print(f"Total parameters       : {total_params:,}")
print(f"Trainable parameters   : {trainable_params:,}")
print(f"Frozen parameters      : {frozen_params:,}")
print(f"Parameter objects      : {len(all_params):,}")


# ------------------------------------------------------------
# 2. Recursive parameter count
#
# IMPORTANT:
# module.parameters() recursively includes parameters from
# child modules.
#
# Therefore:
#   EventEncoder -> includes all Conv2d / GroupNorm parameters
#   Sequential   -> includes all parameters inside it
#
# This is the count you actually want for component analysis.
# ------------------------------------------------------------

def recursive_parameter_stats(module):
    params = list(module.parameters())

    total = sum(p.numel() for p in params)
    trainable = sum(p.numel() for p in params if p.requires_grad)
    frozen = total - trainable

    return total, trainable, frozen


# ------------------------------------------------------------
# 3. Direct parameters only
#
# recurse=False means ONLY parameters directly registered
# inside this particular module.
# ------------------------------------------------------------

def direct_parameter_stats(module):
    params = list(module.parameters(recurse=False))

    total = sum(p.numel() for p in params)
    trainable = sum(p.numel() for p in params if p.requires_grad)
    frozen = total - trainable

    return total, trainable, frozen


# ------------------------------------------------------------
# 4. Top-level component accounting
# ------------------------------------------------------------

print("\n[2] TOP-LEVEL COMPONENT PARAMETER ACCOUNTING")
print("-" * 100)

header = (
    f"{'Component':<25}"
    f"{'Direct':>15}"
    f"{'Recursive':>15}"
    f"{'Trainable':>15}"
    f"{'Frozen':>15}"
    f"{'% Total':>10}"
)

print(header)
print("-" * 100)

top_level_sum = 0

for name, module in model.named_children():

    direct_total, direct_trainable, direct_frozen = \
        direct_parameter_stats(module)

    recursive_total, recursive_trainable, recursive_frozen = \
        recursive_parameter_stats(module)

    top_level_sum += recursive_total

    percentage = (
        100.0 * recursive_total / total_params
        if total_params > 0 else 0.0
    )

    print(
        f"{name:<25}"
        f"{direct_total:>15,}"
        f"{recursive_total:>15,}"
        f"{recursive_trainable:>15,}"
        f"{recursive_frozen:>15,}"
        f"{percentage:>9.2f}%"
    )

print("-" * 100)

print(
    f"{'TOP-LEVEL SUM':<25}"
    f"{'':>15}"
    f"{top_level_sum:>15,}"
    f"{'':>15}"
    f"{'':>15}"
    f"{100.0 * top_level_sum / total_params:>9.2f}%"
)

print(
    f"\nModel total parameters : {total_params:,}"
)
print(
    f"Top-level component sum: {top_level_sum:,}"
)
print(
    f"Difference             : {total_params - top_level_sum:,}"
)

if top_level_sum == total_params:
    print("\nPASS: Top-level component parameter counts exactly match model total.")
else:
    print("\nFAIL: Top-level component counts DO NOT match model total.")


# ------------------------------------------------------------
# 5. Detailed recursive tree
#
# This explains why modules such as EventEncoder itself show
# zero direct parameters but contain millions recursively.
# ------------------------------------------------------------

print("\n\n[3] DETAILED MODULE PARAMETER TREE")
print("-" * 100)

def print_module_tree(module, prefix="", is_root=True):

    children = list(module.named_children())

    for idx, (name, child) in enumerate(children):

        direct_total, direct_trainable, direct_frozen = \
            direct_parameter_stats(child)

        recursive_total, recursive_trainable, recursive_frozen = \
            recursive_parameter_stats(child)

        percentage = (
            100.0 * recursive_total / total_params
            if total_params > 0 else 0.0
        )

        print(
            f"{prefix}{name:<30}"
            f" | {child.__class__.__name__:<30}"
            f" | direct={direct_total:>10,}"
            f" | recursive={recursive_total:>10,}"
            f" | {percentage:6.2f}%"
        )

        print_module_tree(
            child,
            prefix=prefix + "    ",
            is_root=False
        )


print(
    f"{'ROOT':<30}"
    f" | {model.__class__.__name__:<30}"
    f" | direct={direct_parameter_stats(model)[0]:>10,}"
    f" | recursive={total_params:>10,}"
)

print_module_tree(model)


# ------------------------------------------------------------
# 6. Parameter distribution by major component
# ------------------------------------------------------------

print("\n\n[4] PARAMETER DISTRIBUTION")
print("-" * 100)

component_stats = []

for name, module in model.named_children():

    total, trainable, frozen = recursive_parameter_stats(module)

    component_stats.append(
        (name, total, trainable, frozen)
    )

component_stats.sort(key=lambda x: x[1], reverse=True)

for name, total, trainable, frozen in component_stats:

    pct = (
        100.0 * total / total_params
        if total_params > 0 else 0.0
    )

    print(
        f"{name:<25}"
        f"{total:>15,} params"
        f"{pct:>10.2f}%"
    )


# ------------------------------------------------------------
# 7. Parameter sharing check
#
# If the same Parameter object is registered in multiple places,
# summing modules naively can double-count it.
#
# We verify that every Parameter object has a unique identity.
# ------------------------------------------------------------

print("\n\n[5] PARAMETER SHARING CHECK")
print("-" * 100)

parameter_locations = defaultdict(list)

for name, param in model.named_parameters():

    parameter_locations[id(param)].append(name)

shared_parameters = {
    pid: names
    for pid, names in parameter_locations.items()
    if len(names) > 1
}

if len(shared_parameters) == 0:
    print("PASS: No shared Parameter objects detected.")
else:
    print("FAIL: Shared Parameter objects detected!")

    for pid, names in shared_parameters.items():
        print("\nShared parameter:")
        for name in names:
            print(f"    {name}")


# ------------------------------------------------------------
# 8. Parameter finiteness check
# ------------------------------------------------------------

print("\n\n[6] PARAMETER FINITENESS CHECK")
print("-" * 100)

bad_parameters = []

for name, param in model.named_parameters():

    if not torch.isfinite(param).all():
        bad_parameters.append(name)

if len(bad_parameters) == 0:
    print("PASS: All model parameters contain finite values.")
else:
    print("FAIL: Non-finite parameters detected!")

    for name in bad_parameters:
        print(f"    {name}")


# ------------------------------------------------------------
# 9. Buffer finiteness check
#
# Important because BatchNorm contains buffers such as:
# running_mean
# running_var
#
# These are NOT parameters but are part of model state.
# ------------------------------------------------------------

print("\n\n[7] BUFFER FINITENESS CHECK")
print("-" * 100)

bad_buffers = []

for name, buffer in model.named_buffers():

    if torch.is_floating_point(buffer):
        if not torch.isfinite(buffer).all():
            bad_buffers.append(name)

if len(bad_buffers) == 0:
    print("PASS: All floating-point buffers contain finite values.")
else:
    print("FAIL: Non-finite buffers detected!")

    for name in bad_buffers:
        print(f"    {name}")


# ------------------------------------------------------------
# 10. Parameter shapes grouped by top-level component
# ------------------------------------------------------------

print("\n\n[8] PARAMETER SHAPES BY COMPONENT")
print("-" * 100)

for component_name, component in model.named_children():

    print(f"\n{'=' * 80}")
    print(f"COMPONENT: {component_name}")
    print(f"TYPE     : {component.__class__.__name__}")
    print(f"{'=' * 80}")

    found = False

    for param_name, param in component.named_parameters():

        found = True

        print(
            f"{param_name:<65}"
            f"shape={str(tuple(param.shape)):<25}"
            f"numel={param.numel():>10,}"
        )

    if not found:
        print("No parameters registered.")


# ------------------------------------------------------------
# 11. State-dict consistency check
# ------------------------------------------------------------

print("\n\n[9] STATE DICT CHECK")
print("-" * 100)

state_dict = model.state_dict()

parameter_state_entries = 0

for name, _ in model.named_parameters():
    if name in state_dict:
        parameter_state_entries += 1

print(f"Parameter objects       : {len(all_params):,}")
print(f"State-dict entries      : {len(state_dict):,}")
print(f"Parameter state entries : {parameter_state_entries:,}")

if parameter_state_entries == len(all_params):
    print("PASS: Every parameter has a corresponding state-dict entry.")
else:
    print(
        "WARNING: Number of parameter objects and parameter "
        "state entries differ."
    )


# ------------------------------------------------------------
# 12. FINAL ARCHITECTURE VERDICT
# ------------------------------------------------------------

print("\n\n" + "=" * 100)
print("FINAL VERIFICATION")
print("=" * 100)

checks = {
    "Parameter accounting":
        top_level_sum == total_params,

    "Parameter finiteness":
        len(bad_parameters) == 0,

    "Buffer finiteness":
        len(bad_buffers) == 0,

    "No unexpected parameter sharing":
        len(shared_parameters) == 0,
}

all_pass = True

for check_name, passed in checks.items():

    if passed:
        print(f"PASS     {check_name}")
    else:
        print(f"FAIL     {check_name}")
        all_pass = False

print("-" * 100)

if all_pass:
    print("ALL STATIC ARCHITECTURE CHECKS PASSED.")
else:
    print("ONE OR MORE ARCHITECTURE CHECKS FAILED.")

print("=" * 100)

MODEL PARAMETER & ARCHITECTURE VERIFICATION


NameError: name 'model' is not defined